In [ ]:
!nvidia-smi
!find /kaggle/input -maxdepth 4

In [ ]:
%%writefile sgn.py
#!/usr/bin/env python
"""SpectralGate-Net: minimal study on simulated thermal crossover built from COD10K.

  python sgn.py train --mode full   --root COD10K-v3
  python sgn.py train --mode fixed  --root COD10K-v3
  python sgn.py train --mode concat --root COD10K-v3
  python sgn.py train --mode rgb    --root COD10K-v3
  python sgn.py report                      # writes LaTeX tables + PDF plots

  python sgn.py train --mode full --smoke   # 1-minute CPU/GPU sanity check, synthetic data
"""
import argparse, glob, json, math, os, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as Fn
import torchvision
from torchvision.ops import deform_conv2d

DELTAS = [8, 4, 2, 1, 0]            # test-time target-minus-background temperature (K)
TRAIN_D = [0, 0.5, 1, 2, 4, 8]      # training-time contrast values (sampled per image)

# ------------------------------------------------------------------ thermal simulation
H_PL, C_PL, K_PL = 6.626e-34, 2.998e8, 1.381e-23


def planck_band(T):
    """Band-integrated (8-14 um) blackbody radiance, arbitrary units. T in kelvin."""
    lam = torch.linspace(8e-6, 14e-6, 13, dtype=torch.float64, device=T.device).view(1, -1, 1, 1)
    T = T.double()
    w = 2 * H_PL * C_PL ** 2 / lam ** 5 / torch.expm1(H_PL * C_PL / (lam * K_PL * T))
    return w.sum(1, keepdim=True)


def smooth_noise(B, H, W, res, dev, gen=None):
    n = torch.randn(B, 1, res, res, device=dev, generator=gen)
    return Fn.interpolate(n, size=(H, W), mode="bicubic", align_corners=False)


def gauss_blur(x, sigma=1.0, k=5):
    ax = torch.arange(k, device=x.device, dtype=x.dtype) - k // 2
    g = torch.exp(-ax ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    x = Fn.conv2d(Fn.pad(x, (k // 2, k // 2, 0, 0), mode="reflect"), g.view(1, 1, 1, k))
    return Fn.conv2d(Fn.pad(x, (0, 0, k // 2, k // 2), mode="reflect"), g.view(1, 1, k, 1))


@torch.no_grad()
def simulate_lwir(rgb, mask, delta, gen=None, T0=295.0, netd=0.05):
    """rgb (B,3,H,W) in [0,1]; mask (B,1,H,W) in {0,1}; delta (B,) kelvin -> LWIR (B,1,H,W) in [0,1].
    Tbg = T0 + 2K * spatially-correlated field + 2K * (luminance - 0.5)   [scene-dependent term]
    Ttarget = Tbg + delta inside mask; Gaussian PSF (sigma 1 px); NETD 50 mK; fixed 280-320 K -> 8-bit range."""
    B, _, H, W = rgb.shape
    lum = rgb.mean(1, keepdim=True)
    T = T0 + 2.0 * smooth_noise(B, H, W, 12, rgb.device, gen) + 2.0 * (lum - 0.5)
    T = T + delta.view(B, 1, 1, 1).to(rgb.device) * mask
    T = gauss_blur(T)
    T = T + netd * torch.randn(T.shape, device=T.device, generator=gen)
    lo = planck_band(torch.full((1, 1, 1, 1), 280.0, device=T.device))
    hi = planck_band(torch.full((1, 1, 1, 1), 320.0, device=T.device))
    g = ((planck_band(T) - lo) / (hi - lo)).clamp(0, 1).float()
    return torch.round(g * 255) / 255


def shadow_rgb(rgb, gen=None):
    """Dappled-shadow darkening of the visible image."""
    B, _, H, W = rgb.shape
    s = torch.sigmoid(2.0 * smooth_noise(B, H, W, 8, rgb.device, gen))
    return rgb * (0.25 + 0.75 * s)


# ------------------------------------------------------------------ Haar DWT / IDWT
def dwt(x):
    a, b, c, d = x[..., 0::2, 0::2], x[..., 0::2, 1::2], x[..., 1::2, 0::2], x[..., 1::2, 1::2]
    return (a + b + c + d) / 2, (a - b + c - d) / 2, (a + b - c - d) / 2, (a - b - c + d) / 2


def idwt(ll, lh, hl, hh):
    a = (ll + lh + hl + hh) / 2; b = (ll - lh + hl - hh) / 2
    c = (ll + lh - hl - hh) / 2; d = (ll - lh - hl + hh) / 2
    B, C, H, W = ll.shape
    return Fn.pixel_shuffle(torch.stack([a, b, c, d], 2).reshape(B, C * 4, H, W), 2)


# ------------------------------------------------------------------ gate
class SoftEntropy(nn.Module):
    def __init__(self, M=16, tau=0.2):
        super().__init__()
        self.e = nn.Parameter(torch.linspace(-1, 1, M)); self.tau = tau; self.M = M

    def forward(self, t):                      # t in [-1,1], (B,C,H,W)
        v = t.float().reshape(t.shape[0], -1, 1)
        p = torch.softmax(-(v - self.e.view(1, 1, -1)) ** 2 / (2 * self.tau ** 2), dim=-1)
        ph = p.mean(1)
        return -(ph * torch.log(ph + 1e-7)).sum(-1) / math.log(self.M)   # normalised to [0,1]


class Gate(nn.Module):
    def __init__(self, extra=False):
        super().__init__()
        self.ent = SoftEntropy(); self.extra = extra
        self.mlp = nn.Sequential(nn.Linear(3 if extra else 2, 16), nn.GELU(), nn.Linear(16, 1))
        nn.init.zeros_(self.mlp[2].weight); nn.init.zeros_(self.mlp[2].bias)   # start at alpha = 0.5

    def forward(self, f, raw_std=None):
        t = torch.tanh(f.float())
        var = t.var(dim=(2, 3)).mean(1) * 10.0
        z = [var, self.ent(t)]
        if self.extra:
            z.append(raw_std.float())
        return torch.sigmoid(self.mlp(torch.stack(z, 1))).squeeze(1)      # (B,)


# ------------------------------------------------------------------ fusion blocks
class WaveFuse(nn.Module):
    def __init__(self, C, mode, heads=4, extra=False):
        super().__init__()
        self.mode = mode
        self.off = nn.Conv2d(2 * C, 27, 3, padding=1)
        nn.init.zeros_(self.off.weight); nn.init.zeros_(self.off.bias)
        self.w_ll = nn.Parameter(torch.empty(C, C, 3, 3)); nn.init.kaiming_uniform_(self.w_ll, a=math.sqrt(5))
        self.w_hf = nn.Parameter(torch.empty(C, C, 3, 3)); nn.init.kaiming_uniform_(self.w_hf, a=math.sqrt(5))
        self.g_rgb, self.g_ir = Gate(extra), Gate(extra)
        mk = lambda: nn.ModuleList([nn.MultiheadAttention(C, heads, batch_first=True) for _ in range(3)])
        self.r2i, self.i2r = mk(), mk()                       # separate projections per direction
        self.ln = nn.ModuleList([nn.LayerNorm(C) for _ in range(3)])

    def forward(self, fr, fi, raw_r=None, raw_i=None):
        llr, lhr, hlr, hhr = dwt(fr)
        lli, lhi, hli, hhi = dwt(fi)
        with torch.autocast(device_type=fr.device.type, enabled=False):      # deform conv in fp32
            o = self.off(torch.cat([llr, lli], 1).float())
            off, m = o[:, :18], 2 * torch.sigmoid(o[:, 18:])
            lli = deform_conv2d(lli.float(), off, self.w_ll, padding=1, mask=m)
            lhi, hli, hhi = [deform_conv2d(x.float(), off, self.w_hf, padding=1, mask=m) for x in (lhi, hli, hhi)]
        B = fr.shape[0]
        if self.mode == "fixed":
            a_r = a_i = torch.full((B,), 0.5, device=fr.device)
        else:
            a_r, a_i = self.g_rgb(llr, raw_r), self.g_ir(lli, raw_i)
        self.gates = (a_r, a_i)
        ar4, ai4 = a_r.view(-1, 1, 1, 1), a_i.view(-1, 1, 1, 1)
        ll = ar4 * llr + ai4 * lli
        outs = []
        for k, (X, Y) in enumerate([(lhr, lhi), (hlr, hli), (hhr, hhi)]):
            Bc, C, h, w = X.shape
            xt, yt = X.flatten(2).transpose(1, 2), Y.to(X.dtype).flatten(2).transpose(1, 2)
            A1 = self.r2i[k](xt, yt, yt, need_weights=False)[0]                # Q from RGB, K/V from LWIR
            A2 = self.i2r[k](yt, xt, xt, need_weights=False)[0]
            o_ = a_i.view(-1, 1, 1) * A1 + a_r.view(-1, 1, 1) * A2 + self.ln[k](xt * yt)
            outs.append(o_.transpose(1, 2).reshape(Bc, C, h, w))
        return idwt(ll, *outs)


class GatedSum(nn.Module):
    def __init__(self, C, mode, extra=False):
        super().__init__()
        self.mode = mode; self.g_rgb, self.g_ir = Gate(extra), Gate(extra)

    def forward(self, fr, fi, raw_r=None, raw_i=None):
        B = fr.shape[0]
        if self.mode == "fixed":
            a_r = a_i = torch.full((B,), 0.5, device=fr.device)
        else:
            a_r, a_i = self.g_rgb(fr, raw_r), self.g_ir(fi, raw_i)
        self.gates = (a_r, a_i)
        return a_r.view(-1, 1, 1, 1) * fr + a_i.view(-1, 1, 1, 1) * fi


class ConcatFuse(nn.Module):
    def __init__(self, C):
        super().__init__(); self.c = nn.Sequential(nn.Conv2d(2 * C, C, 1), nn.BatchNorm2d(C), nn.ReLU())

    def forward(self, fr, fi, *_):
        return self.c(torch.cat([fr, fi], 1))


# ------------------------------------------------------------------ network
def run_backbone(m, x):
    x = m.maxpool(m.relu(m.bn1(m.conv1(x)))); x = m.layer1(x)
    f3 = m.layer2(x); f4 = m.layer3(f3); f5 = m.layer4(f4)
    return f3, f4, f5


class Net(nn.Module):
    def __init__(self, mode, pretrained=True, extra=False):
        super().__init__()
        self.mode, self.extra = mode, extra
        w = "IMAGENET1K_V1" if pretrained else None
        self.b_rgb = torchvision.models.resnet18(weights=w)
        if mode != "rgb":
            self.b_ir = torchvision.models.resnet18(weights=w)
        ch = (128, 256, 512)
        if mode in ("full", "fixed"):
            self.fuse = nn.ModuleList([WaveFuse(128, mode, extra=extra), WaveFuse(256, mode, extra=extra),
                                       GatedSum(512, mode, extra=extra)])
        elif mode == "concat":
            self.fuse = nn.ModuleList([ConcatFuse(c) for c in ch])
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in ch])
        self.out = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.Conv2d(64, 1, 1))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, rgb, ir):
        size = rgb.shape[-2:]
        fr = run_backbone(self.b_rgb, (rgb - self.mean) / self.std)
        if self.mode == "rgb":
            fused, gates = fr, []
        else:
            fi = run_backbone(self.b_ir, ((ir - 0.45) / 0.225).expand(-1, 3, -1, -1))
            raw_r = rgb.flatten(1).std(1) * 10 if self.extra else None
            raw_i = ir.flatten(1).std(1) * 10 if self.extra else None
            fused = [f(a, b, raw_r, raw_i) for f, a, b in zip(self.fuse, fr, fi)]
            gates = [f.gates for f in self.fuse] if self.mode in ("full", "fixed") else []
        p = [l(f) for l, f in zip(self.lat, fused)]
        up = lambda t: Fn.interpolate(t, size=p[0].shape[-2:], mode="bilinear", align_corners=False)
        x = p[0] + up(p[1]) + up(p[2])
        return Fn.interpolate(self.out(x), size=size, mode="bilinear", align_corners=False), gates


# ------------------------------------------------------------------ data
def load_split(root, split, size):
    """Finds COD10K camouflaged (COD10K-CAM-*) images + GT_Object masks anywhere under root.
    Non-camouflaged / background images are skipped (they have no camouflage target)."""
    cache = f"cache_{split}_{size}.npz"
    if os.path.exists(cache):
        d = np.load(cache); return d["x"], d["m"]
    from PIL import Image
    from concurrent.futures import ThreadPoolExecutor
    in_split = lambda p: split.lower() in [c.lower() for c in p.split(os.sep)]
    ims = sorted(p for p in glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True)
                 if in_split(p) and os.path.basename(p).startswith("COD10K-CAM-"))
    gts = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(root, "**", "*.png"), recursive=True)
           if in_split(p) and "object" in p.lower() and "instance" not in p.lower()}
    ims = [p for p in ims if os.path.splitext(os.path.basename(p))[0] in gts]
    assert ims, f"no COD10K-CAM images+GT_Object masks found for split {split} under {root}"
    print(f"{split}: {len(ims)} camouflaged images", flush=True)

    def ld(p):
        stem = os.path.splitext(os.path.basename(p))[0]
        x = np.asarray(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))
        m = np.asarray(Image.open(gts[stem]).convert("L").resize((size, size), Image.NEAREST)) > 127
        return x, m

    with ThreadPoolExecutor(8) as ex:
        r = list(ex.map(ld, ims))
    x = np.stack([a for a, _ in r]); m = np.stack([b for _, b in r]).astype(np.uint8)
    np.savez(cache, x=x, m=m)
    return x, m


def synthetic(n, size):
    x = np.random.randint(0, 255, (n, size, size, 3), dtype=np.uint8)
    m = np.zeros((n, size, size), np.uint8)
    for i in range(n):
        a, b = np.random.randint(8, size // 2, 2); m[i, a:a + size // 4, b:b + size // 4] = 1
    return x, m


def get_batch(x, m, idx, dev):
    rgb = torch.from_numpy(x[idx]).to(dev).permute(0, 3, 1, 2).float() / 255
    mk = torch.from_numpy(m[idx]).to(dev).unsqueeze(1).float()
    return rgb, mk


# ------------------------------------------------------------------ train / eval
def loss_fn(logits, mk, gates, mode):
    bce = Fn.binary_cross_entropy_with_logits(logits, mk)
    p = torch.sigmoid(logits)
    iou = 1 - ((p * mk).sum((1, 2, 3)) + 1) / ((p + mk - p * mk).sum((1, 2, 3)) + 1)
    loss = bce + iou.mean()
    if mode == "full" and gates:
        lg = sum(-0.5 * ((ar - 0.5) ** 2 + (ai - 0.5) ** 2).mean() for ar, ai in gates) / len(gates)
        loss = loss + 0.05 * lg
    return loss


@torch.no_grad()
def evaluate(model, x, m, dev, bs=32):
    model.eval(); res = {}
    for s in (0, 1):
        for d in DELTAS:
            gen = torch.Generator(device=dev).manual_seed(1234)
            iou = f1 = mae = n = 0.0; ar = [0.0] * 3; ai = [0.0] * 3
            for i in range(0, len(x), bs):
                idx = np.arange(i, min(i + bs, len(x)))
                rgb, mk = get_batch(x, m, idx, dev)
                ir = simulate_lwir(rgb, mk, torch.full((len(idx),), float(d)), gen)
                if s:
                    rgb = shadow_rgb(rgb, gen)
                logits, gates = model(rgb, ir)
                p = torch.sigmoid(logits); pb = (p > 0.5).float()
                inter = (pb * mk).sum((1, 2, 3)); uni = (pb + mk - pb * mk).sum((1, 2, 3))
                iou += (inter / (uni + 1e-6)).sum().item()
                f1 += (2 * inter / (pb.sum((1, 2, 3)) + mk.sum((1, 2, 3)) + 1e-6)).sum().item()
                mae += (p - mk).abs().mean((1, 2, 3)).sum().item(); n += len(idx)
                for j, (a_r, a_i) in enumerate(gates):
                    ar[j] += a_r.sum().item(); ai[j] += a_i.sum().item()
            res[f"d{d}_s{s}"] = dict(iou=iou / n, f1=f1 / n, mae=mae / n,
                                     alpha_rgb=[v / n for v in ar], alpha_ir=[v / n for v in ai])
            print(f"  delta={d} shadow={s}  IoU={iou/n:.4f}  F1={f1/n:.4f}  MAE={mae/n:.4f}  alpha_ir={[round(v/n,3) for v in ai]}", flush=True)
    return res


@torch.no_grad()
def latency(model, size, dev):
    model.eval(); x = torch.rand(1, 3, size, size, device=dev); y = torch.rand(1, 1, size, size, device=dev)
    for _ in range(10): model(x, y)
    if dev.type == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(50): model(x, y)
    if dev.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / 50 * 1000


def train(a):
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if a.smoke:
        a.size, a.epochs, a.bs = 96, 1, 8
        xtr, mtr = synthetic(32, a.size); xte, mte = synthetic(16, a.size)
    else:
        xtr, mtr = load_split(a.root, "Train", a.size); xte, mte = load_split(a.root, "Test", a.size)
    print(f"train {len(xtr)}  test {len(xte)}  device {dev}", flush=True)
    model = Net(a.mode, pretrained=not a.smoke, extra=a.raw_desc).to(dev)
    nparam = sum(p.numel() for p in model.parameters()) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    steps = a.epochs * (len(xtr) // a.bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=steps, pct_start=0.1)
    use_amp = dev.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    t0 = time.time(); it = 0
    for ep in range(a.epochs):
        model.train(); perm = np.random.permutation(len(xtr))
        for i in range(0, len(xtr) - a.bs + 1, a.bs):
            idx = np.sort(perm[i:i + a.bs])
            rgb, mk = get_batch(xtr, mtr, idx, dev)
            if random.random() < 0.5:
                rgb, mk = rgb.flip(-1), mk.flip(-1)
            delta = torch.tensor(np.random.choice(TRAIN_D, a.bs), dtype=torch.float32)
            ir = simulate_lwir(rgb, mk, delta)
            sh = (torch.rand(a.bs, device=dev) < 0.3).view(-1, 1, 1, 1)
            rgb = torch.where(sh, shadow_rgb(rgb), rgb)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                logits, gates = model(rgb, ir)
            loss = loss_fn(logits.float(), mk, gates, a.mode)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); it += 1
            if it % 50 == 0:
                print(f"ep {ep} it {it}/{steps} loss {loss.item():.4f}  {time.time()-t0:.0f}s", flush=True)
    print("evaluating ...", flush=True)
    res = evaluate(model, xte, mte, dev)
    out = dict(mode=a.mode, seed=a.seed, epochs=a.epochs, size=a.size, params_M=nparam,
               latency_ms_bs1=latency(model, a.size, dev), device=str(torch.cuda.get_device_name(0) if use_amp else "cpu"),
               raw_desc=a.raw_desc, res=res)
    name = f"results_{a.mode}{'_raw' if a.raw_desc else ''}_seed{a.seed}.json"
    json.dump(out, open("smoke_" + name if a.smoke else name, "w"), indent=1)
    print("saved", name)


# ------------------------------------------------------------------ report
def report():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    R = {}
    for f in glob.glob("results_*_seed*.json"):
        d = json.load(open(f)); R.setdefault(d["mode"] + ("_raw" if d.get("raw_desc") else ""), []).append(d)
    names = {"rgb": "RGB only", "concat": "Early concatenation", "fixed": r"Fixed gates ($\alpha=0.5$)",
             "full": "SpectralGate-Net", "full_raw": "SpectralGate-Net + raw descriptor"}
    order = [k for k in names if k in R]

    def st(k, key, fld="iou"):
        v = np.array([r["res"][key][fld] for r in R[k]]) * 100
        return v.mean(), v.std()

    cell = lambda k, key: (f"{st(k,key)[0]:.1f}" if len(R[k]) == 1 else f"{st(k,key)[0]:.1f}$\\pm${st(k,key)[1]:.1f}")
    for s, tag in ((0, "clean"), (1, "shadow")):
        lines = [r"\begin{tabular}{l" + "c" * len(DELTAS) + "}", r"\toprule",
                 "Method & " + " & ".join(rf"$\delta={d}$" for d in DELTAS) + r" \\", r"\midrule"]
        for k in order:
            lines.append(names[k] + " & " + " & ".join(cell(k, f"d{d}_s{s}") for d in DELTAS) + r" \\")
        lines += [r"\bottomrule", r"\end{tabular}"]
        open(f"table_sim_{tag}.tex", "w").write("\n".join(lines))
    eff = [r"\begin{tabular}{lcc}", r"\toprule", r"Method & Params (M) & Latency (ms, bs=1) \\", r"\midrule"]
    for k in order:
        eff.append(f"{names[k]} & {R[k][0]['params_M']:.1f} & {R[k][0]['latency_ms_bs1']:.1f} \\\\")
    eff += [r"\bottomrule", r"\end{tabular}"]
    open("table_eff.tex", "w").write("\n".join(eff))
    print("device:", R[order[0]][0]["device"])

    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    for s in (0, 1):
        for k in order:
            ax[s].plot(DELTAS, [st(k, f"d{d}_s{s}")[0] for d in DELTAS], marker="o", label=k)
        ax[s].set_xlabel(r"thermal contrast $\delta$ (K)"); ax[s].set_ylabel("mIoU (%)")
        ax[s].set_title("clean RGB" if s == 0 else "RGB with shadow"); ax[s].invert_xaxis()
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig("fig_iou_vs_delta.pdf")

    gk = [k for k in ("full", "full_raw") if k in R]
    if gk:
        fig, ax = plt.subplots(figsize=(4, 3))
        for k in gk:
            for s in (0, 1):
                for j, stg in enumerate(("P3", "P4", "P5")):
                    y = [np.mean([r["res"][f"d{d}_s{s}"]["alpha_ir"][j] for r in R[k]]) for d in DELTAS]
                    ax.plot(DELTAS, y, marker="o", ls="-" if s == 0 else "--", label=f"{k} {stg} {'shadow' if s else 'clean'}")
        ax.set_xlabel(r"$\delta$ (K)"); ax.set_ylabel(r"mean $\alpha_{ir}$"); ax.invert_xaxis(); ax.legend(fontsize=5)
        plt.tight_layout(); plt.savefig("fig_alpha_vs_delta.pdf")
    print("wrote table_sim_clean.tex table_sim_shadow.tex table_eff.tex fig_iou_vs_delta.pdf fig_alpha_vs_delta.pdf")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["train", "report"])
    ap.add_argument("--mode", default="full", choices=["rgb", "concat", "fixed", "full"])
    ap.add_argument("--root", default="COD10K-v3")
    ap.add_argument("--size", type=int, default=352)
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--raw_desc", action="store_true", help="add raw-input contrast as 3rd gate feature")
    ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    train(a) if a.cmd == "train" else report()


In [ ]:
!python sgn.py train --mode full --smoke

In [ ]:
!python sgn.py train --mode full --root /kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3

In [ ]:
%%bash
R=/kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3
cd /kaggle/working

# Launch GPU 0 queue (3 wavelet runs)
nohup sh -c "CUDA_VISIBLE_DEVICES=0 python sgn.py train --mode fixed  --root $R; \
             CUDA_VISIBLE_DEVICES=0 python sgn.py train --mode full  --seed 1 --root $R; \
             CUDA_VISIBLE_DEVICES=0 python sgn.py train --mode fixed --seed 1 --root $R" > gpu0.log 2>&1 &
PID0=$!

# Launch GPU 1 queue (4 baseline runs)
nohup sh -c "CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode rgb    --root $R; \
             CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode concat --root $R; \
             CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode rgb    --seed 1 --root $R; \
             CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode concat --seed 1 --root $R" > gpu1.log 2>&1 &
PID1=$!

echo "Started GPU 0 (PID $PID0) and GPU 1 (PID $PID1). Monitoring every 30s..."

# Print live progress until both GPUs finish
while kill -0 $PID0 2>/dev/null || kill -0 $PID1 2>/dev/null; do
    sleep 30
    echo "=== [$(date +%H:%M:%S)] Status ==="
    echo "[GPU 0]: $(tail -n 1 gpu0.log 2>/dev/null)"
    echo "[GPU 1]: $(tail -n 1 gpu1.log 2>/dev/null)"
done

echo "All 8 runs complete! Generating LaTeX tables and PDF figures..."
python sgn.py report
head -n 50 table_sim_clean.tex table_sim_shadow.tex table_eff.tex

In [ ]:
%cd /kaggle/working
!ls -la sgn.py results_*.json
!grep -c lgate sgn.py
!ps aux | grep "sgn.py" | grep -v grep | wc -l
!tail -n 2 gpu0.log gpu1.log

In [ ]:
%cd /kaggle/working
s = open('sgn.py').read()
def rep(a, b):
    global s
    assert a in s, "NOT FOUND: " + a
    s = s.replace(a, b, 1)
rep('def loss_fn(logits, mk, gates, mode):', 'def loss_fn(logits, mk, gates, mode, lgate=0.05):')
rep('loss = loss + 0.05 * lg', 'loss = loss + lgate * lg')
rep('loss = loss_fn(logits.float(), mk, gates, a.mode)', 'loss = loss_fn(logits.float(), mk, gates, a.mode, a.lgate)')
rep('raw_desc=a.raw_desc, res=res)', 'raw_desc=a.raw_desc, lgate=a.lgate, res=res)')
rep("""name = f"results_{a.mode}{'_raw' if a.raw_desc else ''}_seed{a.seed}.json\"""",
    """name = f"results_{a.mode}{'_raw' if a.raw_desc else ''}{'_nolg' if a.lgate == 0 else ''}_seed{a.seed}.json\"""")
rep('ap.add_argument("--smoke", action="store_true")',
    'ap.add_argument("--lgate", type=float, default=0.05)\n    ap.add_argument("--smoke", action="store_true")')
rep('R.setdefault(d["mode"] + ("_raw" if d.get("raw_desc") else ""), []).append(d)',
    'R.setdefault(d["mode"] + ("_raw" if d.get("raw_desc") else "") + ("_nolg" if d.get("lgate", 0.05) == 0 else ""), []).append(d)')
rep('"full": "SpectralGate-Net", "full_raw": "SpectralGate-Net + raw descriptor"}',
    '"full": "SpectralGate-Net", "full_nolg": r"SpectralGate-Net, no $L_{gate}$",\n             "full_raw": "SpectralGate-Net + raw descriptor", "full_raw_nolg": r"SpectralGate-Net + raw desc., no $L_{gate}$"}')
rep('gk = [k for k in ("full", "full_raw") if k in R]',
    'gk = [k for k in ("full", "full_nolg", "full_raw", "full_raw_nolg") if k in R]')
open('sgn.py', 'w').write(s)
print("patched")

In [ ]:
%%bash
cd /kaggle/working
R=/kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3
# GPU 1 is free now: no polarisation loss
nohup sh -c "CUDA_VISIBLE_DEVICES=1 python -u sgn.py train --mode full --lgate 0 --root $R" > gpu1b.log 2>&1 &
# GPU 0: wait for the fixed-seed1 job to end, then run no-L_gate + raw descriptor
nohup sh -c "while pgrep -f '[s]gn.py train --mode fixed --seed 1' >/dev/null; do sleep 10; done; CUDA_VISIBLE_DEVICES=0 python -u sgn.py train --mode full --lgate 0 --raw_desc --root $R" > gpu0b.log 2>&1 &
sleep 2; echo launched

In [ ]:
!python sgn.py report
!cat table_sim_clean.tex table_sim_shadow.tex

In [ ]:
!tail -n 1 gpu0.log gpu0b.log gpu1b.log
!ls results_*.json

In [ ]:
import json, glob
D = [8, 4, 2, 1, 0]
for f in sorted(glob.glob('results_*.json')):
    r = json.load(open(f))['res']
    print(f.replace('results_', '').replace('.json', '').ljust(22),
          'IoU%', [round(r[f'd{d}_s0']['iou'] * 100, 1) for d in D],
          '| alpha_ir d8:', [round(a, 2) for a in r['d8_s0']['alpha_ir']],
          'd0:', [round(a, 2) for a in r['d0_s0']['alpha_ir']])

In [ ]:
!ls results_*.json
!echo "---- gpu1b.log (no L_gate) ----"; tail -n 12 gpu1b.log
!echo "---- gpu0b.log (no L_gate + raw desc) ----"; tail -n 12 gpu0b.log
!ps aux | grep "sgn.py" | grep -v grep | wc -l

In [ ]:
!python sgn.py report
import json, glob
D = [8, 4, 2, 1, 0]
for f in sorted(glob.glob('results_*.json')):
    r = json.load(open(f))['res']
    print(f.replace('results_','').replace('.json','').ljust(24),
          'IoU%', [round(r[f'd{d}_s0']['iou']*100,1) for d in D],
          '| alpha_ir d8:', [round(a,2) for a in r['d8_s0']['alpha_ir']],
          'd0:', [round(a,2) for a in r['d0_s0']['alpha_ir']])

In [ ]:
import glob, zipfile
with zipfile.ZipFile('paper_outputs.zip', 'w') as z:
    for p in glob.glob('*.json') + glob.glob('*.tex') + glob.glob('*.pdf') + glob.glob('*.log') + ['sgn.py']:
        if 'smoke' not in p:
            z.write(p)

In [ ]:
import json, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
D = [8, 4, 2, 1, 0]
runs = {r'with $L_{gate}$': 'results_full_seed0.json', r'no $L_{gate}$': 'results_full_nolg_seed0.json',
        r'no $L_{gate}$ + raw desc.': 'results_full_raw_nolg_seed0.json'}
fig, ax = plt.subplots(1, 3, figsize=(9, 2.6), sharey=True)
for j, st in enumerate(['P3', 'P4', 'P5']):
    for n, f in runs.items():
        r = json.load(open(f))['res']
        ax[j].plot(D, [r[f'd{d}_s0']['alpha_ir'][j] for d in D], marker='o', label=n)
    ax[j].set_title(st); ax[j].set_xlabel(r'$\delta$ (K)'); ax[j].invert_xaxis()
ax[0].set_ylabel(r'mean $\alpha_{ir}$'); ax[0].legend(fontsize=6); plt.tight_layout(); plt.savefig('fig_alpha_clean.pdf')

In [ ]:
import torch
from sgn import dwt, idwt
x = torch.randn(2, 8, 16, 16)
print((idwt(*dwt(x)) - x).abs().max().item())
print(abs(sum((b**2).sum() for b in dwt(x)).item() - (x**2).sum().item()))

In [ ]:
import json
for f in ['results_full_seed0.json', 'results_full_seed1.json']:
    print(f, [round(a, 2) for a in json.load(open(f))['res']['d8_s0']['alpha_rgb']])

In [ ]:
!pip list 2>/dev/null | grep -i -E "^(torch|torchvision|numpy|pillow|matplotlib) "

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 1. Import your existing modules directly from sgn.py
# (Make sure sgn.py is in your Kaggle working directory)
from sgn import (
    COD10KDataset, 
    simulate_thermal,          # Your paper's Planck/Gaussian simulation function
    SpectralGateNet,
    build_val_loader           # Or your existing validation DataLoader
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Helper Functions: Metric Definitions for Task 1
# -------------------------------------------------------------------------
def local_spatial_std(x, window_size=7):
    """
    Computes local standard deviation across sliding spatial windows.
    x shape: [B, C, H, W]
    """
    c = x.shape[1]
    padding = window_size // 2
    kernel = torch.ones((c, 1, window_size, window_size), device=x.device) / (window_size ** 2)
    
    mean = F.conv2d(x, kernel, padding=padding, groups=c)
    mean_sq = F.conv2d(x ** 2, kernel, padding=padding, groups=c)
    variance = torch.clamp(mean_sq - mean ** 2, min=1e-8)
    return torch.sqrt(variance).mean().item()

def gradient_energy(x):
    """
    Computes average Sobel edge gradient magnitude.
    x shape: [B, C, H, W]
    """
    c = x.shape[1]
    sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=x.device).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=x.device).view(1, 1, 3, 3)
    
    sobel_x = sobel_x.repeat(c, 1, 1, 1)
    sobel_y = sobel_y.repeat(c, 1, 1, 1)
    
    gx = F.conv2d(x, sobel_x, padding=1, groups=c)
    gy = F.conv2d(x, sobel_y, padding=1, groups=c)
    return torch.sqrt(gx ** 2 + gy ** 2 + 1e-8).mean().item()

def extract_f_ll(feature_map):
    """
    Haar Low-Frequency approximation band (LL) via 2x2 average pooling.
    """
    return F.avg_pool2d(feature_map, kernel_size=2, stride=2) * 2.0

# -------------------------------------------------------------------------
# Diagnostic Execution Loop
# -------------------------------------------------------------------------
def run_task1_diagnostic(model, val_loader):
    model.eval().to(device)
    backbone_ir = model.backbone_ir  # Your pretrained ResNet-18 IR stream
    
    deltas = [8, 4, 2, 1, 0]
    
    raw_stds, raw_grads = [], []
    feat_stds, feat_grads = [], []
    
    # Grab one batch of test/val data
    rgb_batch, mask_batch = next(iter(val_loader))
    rgb_batch = rgb_batch.to(device)
    mask_batch = mask_batch.to(device)
    
    with torch.no_grad():
        for d in deltas:
            # 1. Generate thermal image using your exact simulation at delta = d
            lwir_raw = simulate_thermal(rgb_batch, mask_batch, delta=d).to(device)
            if lwir_raw.dim() == 3:
                lwir_raw = lwir_raw.unsqueeze(1)
            if lwir_raw.shape[1] == 1:
                # Expand to 3 channels matching ResNet input convention
                lwir_in = lwir_raw.repeat(1, 3, 1, 1)
            else:
                lwir_in = lwir_raw
            
            # --- Measure Raw LWIR Image Stats ---
            raw_stds.append(local_spatial_std(lwir_raw[:, :1, :, :]))
            raw_grads.append(gradient_energy(lwir_raw[:, :1, :, :]))
            
            # --- Forward Pass Through Backbone to Stage 3 (Post-BN) ---
            x = backbone_ir.conv1(lwir_in)
            x = backbone_ir.bn1(x)
            x = backbone_ir.relu(x)
            x = backbone_ir.maxpool(x)
            x = backbone_ir.layer1(x)
            feat_stage3 = backbone_ir.layer2(x)  # Stage 3 post-BN features
            
            f_ll = extract_f_ll(feat_stage3)
            
            # --- Measure Post-BN Feature Stats ---
            feat_stds.append(local_spatial_std(f_ll))
            feat_grads.append(gradient_energy(f_ll))
            
    # Normalize curves relative to delta=8 (1.0 = baseline at high contrast)
    def rel(arr):
        arr = np.array(arr)
        return arr / (arr[0] + 1e-8)

    # -------------------------------------------------------------------------
    # Plotting
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    
    axes[0].plot(deltas, rel(raw_stds), 'ro-', label='Raw LWIR (Input)')
    axes[0].plot(deltas, rel(feat_stds), 'bs--', label='Post-BN Feature ($F_{LL}$)')
    axes[0].set_title('Normalized Local Std vs $\delta$')
    axes[0].set_xlabel('Thermal Contrast $\delta$ (K)')
    axes[0].set_ylabel('Relative Value (Norm to $\delta=8$)')
    axes[0].grid(True, linestyle=':', alpha=0.6)
    axes[0].legend()

    axes[1].plot(deltas, rel(raw_grads), 'ro-', label='Raw LWIR (Input)')
    axes[1].plot(deltas, rel(feat_grads), 'bs--', label='Post-BN Feature ($F_{LL}$)')
    axes[1].set_title('Normalized Gradient Energy vs $\delta$')
    axes[1].set_xlabel('Thermal Contrast $\delta$ (K)')
    axes[1].set_ylabel('Relative Value (Norm to $\delta=8$)')
    axes[1].grid(True, linestyle=':', alpha=0.6)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('results/fig_diagnostic_raw_vs_bn.png', dpi=300)
    plt.show()

# Run the test
# run_task1_diagnostic(model, val_loader)

In [ ]:
import os
import sys

# 1. Search for sgn.py inside Kaggle workspace
target_dir = None
for root, dirs, files in os.walk('/kaggle'):
    if 'sgn.py' in files:
        target_dir = root
        break

# 2. If not found, clone your repository directly
# if target_dir is None:
print("sgn.py not found. Cloning repo from GitHub...")
!git clone https://github.com/atulsingh369/SpectralGate-Net.git
target_dir = '/kaggle/working/SpectralGate-Net'

# 3. Add to Python path
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

print(f"Loaded module directory: {target_dir}")

In [ ]:
!grep -E "^(class|def) " /kaggle/working/SpectralGate-Net/sgn.py

In [ ]:
import sys
if '/kaggle/working/SpectralGate-Net' not in sys.path:
    sys.path.insert(0, '/kaggle/working/SpectralGate-Net')

import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Import your paper's exact simulation and wavelet functions
from sgn import simulate_lwir, dwt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# 1. Metric Calculations
# -------------------------------------------------------------------------
def local_spatial_std(x, window_size=7):
    c = x.shape[1]
    padding = window_size // 2
    kernel = torch.ones((c, 1, window_size, window_size), device=x.device) / (window_size ** 2)
    mean = F.conv2d(x, kernel, padding=padding, groups=c)
    mean_sq = F.conv2d(x ** 2, kernel, padding=padding, groups=c)
    variance = torch.clamp(mean_sq - mean ** 2, min=1e-8)
    return torch.sqrt(variance).mean().item()

def gradient_energy(x):
    c = x.shape[1]
    sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=x.device).view(1, 1, 3, 3).repeat(c, 1, 1, 1)
    sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=x.device).view(1, 1, 3, 3).repeat(c, 1, 1, 1)
    gx = F.conv2d(x, sobel_x, padding=1, groups=c)
    gy = F.conv2d(x, sobel_y, padding=1, groups=c)
    return torch.sqrt(gx ** 2 + gy ** 2 + 1e-8).mean().item()

# -------------------------------------------------------------------------
# 2. Load Real Validation Samples from COD10K
# -------------------------------------------------------------------------
root_dataset = "/kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3"
img_files = sorted(glob.glob(f"{root_dataset}/**/Image/*.jpg", recursive=True))

if len(img_files) > 0:
    print(f"Found {len(img_files)} images. Loading a batch of 16...")
    batch_imgs, batch_masks = [], []
    for img_path in img_files[:16]:
        mask_path = img_path.replace('/Image/', '/GT_Object/').replace('.jpg', '.png')
        if not os.path.exists(mask_path):
            mask_path = img_path.replace('/Image/', '/GT_Object/').replace('.jpg', '.png')
        
        im = Image.open(img_path).convert('RGB').resize((352, 352))
        mk = Image.open(mask_path).convert('L').resize((352, 352)) if os.path.exists(mask_path) else Image.new('L', (352, 352), 0)
        
        batch_imgs.append(transforms.ToTensor()(im))
        batch_masks.append((transforms.ToTensor()(mk) > 0.5).float())

    rgb_batch = torch.stack(batch_imgs).to(device)
    mask_batch = torch.stack(batch_masks).to(device)
else:
    print("Warning: Dataset directory structure differed, using synthetic sample...")
    rgb_batch = torch.rand(16, 3, 352, 352, device=device) * 0.4 + 0.3
    mask_batch = torch.zeros(16, 1, 352, 352, device=device)
    mask_batch[:, :, 100:250, 100:250] = 1.0

# -------------------------------------------------------------------------
# 3. Model Setup: ResNet-18 Backbone
# -------------------------------------------------------------------------
weights = models.ResNet18_Weights.DEFAULT
backbone_ir = models.resnet18(weights=weights).to(device)
backbone_ir.eval()

# -------------------------------------------------------------------------
# 4. Sweep Thermal Contrast Delta
# -------------------------------------------------------------------------
deltas = [8, 4, 2, 1, 0]
raw_stds, raw_grads = [], []
feat_stds, feat_grads = [], []

print("Running sweep across delta in [8, 4, 2, 1, 0]...")
with torch.no_grad():
    for d in deltas:
        # Call your paper's exact simulation function
        d_tensor = torch.full((rgb_batch.size(0),), float(d), device=rgb_batch.device)
        lwir_raw = simulate_lwir(rgb_batch, mask_batch, delta=d_tensor)
        if lwir_raw.dim() == 3:
            lwir_raw = lwir_raw.unsqueeze(1)
            
        # Metric 1: Raw Image Statistics
        raw_stds.append(local_spatial_std(lwir_raw))
        raw_grads.append(gradient_energy(lwir_raw))
        
        # Expand 1-channel LWIR to 3 channels for ResNet
        lwir_in = lwir_raw.repeat(1, 3, 1, 1) if lwir_raw.shape[1] == 1 else lwir_raw
        
        # Metric 2: Post-BN Stage 3 Feature Statistics
        x = backbone_ir.conv1(lwir_in)
        x = backbone_ir.bn1(x)
        x = backbone_ir.relu(x)
        x = backbone_ir.maxpool(x)
        x = backbone_ir.layer1(x)
        feat_stage3 = backbone_ir.layer2(x) # Post-BN activations
        
        # Decompose using your dwt function -> returns (ll, lh, hl, hh)
        f_ll, _, _, _ = dwt(feat_stage3)
        
        feat_stds.append(local_spatial_std(f_ll))
        feat_grads.append(gradient_energy(f_ll))

def norm(arr):
    arr = np.array(arr)
    return arr / (arr[0] + 1e-8)

# -------------------------------------------------------------------------
# 5. Plotting Curves (Raw strings r'...' prevent SyntaxWarnings)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(deltas, norm(raw_stds), 'ro-', linewidth=2, label='Raw LWIR Image')
axes[0].plot(deltas, norm(feat_stds), 'bs--', linewidth=2, label=r'Post-BN Feature ($F_{LL}$)')
axes[0].set_title(r'Normalized Local Std vs $\delta$')
axes[0].set_xlabel(r'Thermal Contrast $\delta$ (K)')
axes[0].set_ylabel(r'Relative Value (Norm to $\delta=8$)')
axes[0].invert_xaxis()
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend()

axes[1].plot(deltas, norm(raw_grads), 'ro-', linewidth=2, label='Raw LWIR Image')
axes[1].plot(deltas, norm(feat_grads), 'bs--', linewidth=2, label=r'Post-BN Feature ($F_{LL}$)')
axes[1].set_title(r'Normalized Gradient Energy vs $\delta$')
axes[1].set_xlabel(r'Thermal Contrast $\delta$ (K)')
axes[1].set_ylabel(r'Relative Value (Norm to $\delta=8$)')
axes[1].invert_xaxis()
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend()

plt.tight_layout()
os.makedirs('/kaggle/working/results', exist_ok=True)
plt.savefig('/kaggle/working/results/diagnostic_raw_vs_feature.png', dpi=300)
plt.show()

print("Task 1 complete. Saved figure to /kaggle/working/results/diagnostic_raw_vs_feature.png")

In [ ]:
import random
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

def local_std_pi(x, w=7):
    c = x.shape[1]; k = torch.ones((c,1,w,w), device=x.device)/(w*w)
    m = F.conv2d(x, k, padding=w//2, groups=c)
    v = torch.clamp(F.conv2d(x**2, k, padding=w//2, groups=c) - m**2, min=1e-8)
    return torch.sqrt(v).mean(dim=(1,2,3))          # one value per image

def grad_energy_pi(x):
    c = x.shape[1]
    sx = torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]], device=x.device).view(1,1,3,3).repeat(c,1,1,1)
    sy = torch.tensor([[-1.,-2,-1],[0,0,0],[1,2,1]], device=x.device).view(1,1,3,3).repeat(c,1,1,1)
    gx, gy = F.conv2d(x, sx, padding=1, groups=c), F.conv2d(x, sy, padding=1, groups=c)
    return torch.sqrt(gx**2 + gy**2 + 1e-8).mean(dim=(1,2,3))

random.seed(0)
pairs = []
for p in random.sample(img_files, 600):
    m = p.replace('/Image/', '/GT_Object/').replace('.jpg', '.png')
    if os.path.exists(m): pairs.append((p, m))
pairs = pairs[:300]
print("images with masks:", len(pairs))              # must be ~300, else fix paths

# Use your TRAINED IR backbone here if you can load it (adapt the keys to your checkpoint).
# bb = backbone_ir  # fallback: ImageNet weights

deltas = [8, 4, 2, 1, 0]
S = {k: np.zeros((len(deltas), len(pairs))) for k in ['raw_std','raw_gr','ft_std','ft_gr']}

with torch.no_grad():
    for i0 in range(0, len(pairs), 25):
        chunk = pairs[i0:i0+25]
        rgb = torch.stack([transforms.ToTensor()(Image.open(p).convert('RGB').resize((352,352))) for p,_ in chunk]).to(device)
        mk  = torch.stack([(transforms.ToTensor()(Image.open(m).convert('L').resize((352,352)))>0.5).float() for _,m in chunk]).to(device)
        for j, d in enumerate(deltas):
            lw = simulate_lwir(rgb, mk, delta=torch.full((rgb.size(0),), float(d), device=device))
            if lw.dim() == 3: lw = lw.unsqueeze(1)
            sl = slice(i0, i0+len(chunk))
            S['raw_std'][j, sl] = local_std_pi(lw).cpu().numpy()
            S['raw_gr'][j, sl]  = grad_energy_pi(lw).cpu().numpy()
            x = lw.repeat(1,3,1,1) if lw.shape[1]==1 else lw
            x = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(x)   # fix #4
            x = backbone_ir.layer2(backbone_ir.layer1(backbone_ir.maxpool(backbone_ir.relu(backbone_ir.bn1(backbone_ir.conv1(x))))))
            ll = dwt(x)[0]
            S['ft_std'][j, sl] = local_std_pi(ll).cpu().numpy()
            S['ft_gr'][j, sl]  = grad_energy_pi(ll).cpu().numpy()

for k, v in S.items():
    ratio = v / v[0:1]                                # paired: each image vs its own delta=8
    ci = 1.96 * ratio.std(1) / np.sqrt(v.shape[1])
    # unpaired: can ONE image's statistic tell delta=0 from delta=8?
    auc = roc_auc_score(np.r_[np.ones(v.shape[1]), np.zeros(v.shape[1])], np.r_[v[0], v[-1]])
    rho = spearmanr(np.repeat(deltas, v.shape[1]), v.ravel())[0]
    print(f"{k}: paired ratio {np.round(ratio.mean(1),3)} +/- {np.round(ci,3)} | AUC(d=8 vs 0)={auc:.3f} | Spearman={rho:.3f}")

In [ ]:
%%writefile /kaggle/working/phase1.py

"""
Phase 1 experiments (COD10K + simulated LWIR from your sgn.py).

Configs
  rgb_only        RGB backbone only
  concat          two backbones, 1x1-conv concat fusion (paper baseline)
  concat_drop     concat + modality dropout (thermal image blacked out, p=0.25)
  concat_drop_d0  concat + "crossover dropout" (delta forced to 0, p=0.25)
  gate_raw        concat + thermal gate fed by RAW-LWIR statistics, seg loss only
  gate_sup        gate_raw + gate supervised by a delta-derived target
  gate_sup_drop   gate_sup + modality dropout
Extra (--rule): threshold-rule baseline and an ORACLE switch (upper bound for any gate),
built from the trained rgb_only and concat checkpoints.

Usage
  python -u phase1.py --smoke                 # ~2 min check of every code path
  python -u phase1.py --smoke --rule
  python -u phase1.py --configs rgb_only,concat,concat_drop,concat_drop_d0 --seeds 0,1,2
  python -u phase1.py --configs gate_raw,gate_sup,gate_sup_drop --seeds 0,1,2
  python -u phase1.py --rule --seeds 0,1,2
  python -u phase1.py --report
"""
import os, sys, glob, json, time, random, argparse
sys.path.insert(0, '/kaggle/working/SpectralGate-Net')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvm
from PIL import Image

ROOT = '/kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3'
BASE = '/kaggle/working'
CACHE = f'{BASE}/phase1_cache'
SIZE = 352
DELTAS_TEST = [8, 4, 2, 1, 0]
DELTAS_TRAIN = [0, 0.5, 1, 2, 4, 8]
EVAL_SEED = 12345
LAMBDA_SUP, K_SIG, D0_SIG = 1.0, 4.0, 0.5     # gate-target sigmoid(K*(delta-D0)); fixed a priori, do NOT tune on test
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEV == 'cuda'
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

CONFIGS = {
    'rgb_only':       dict(fusion='none'),
    'concat':         dict(fusion='concat'),
    'concat_drop':    dict(fusion='concat', drop=0.25, mode='zero'),
    'concat_drop_d0': dict(fusion='concat', drop=0.25, mode='d0'),
    'gate_raw':       dict(fusion='gate'),
    'gate_sup':       dict(fusion='gate', sup=True),
    'gate_sup_drop':  dict(fusion='gate', sup=True, drop=0.25, mode='zero'),
}


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ----------------------------------------------------------------------------- data
def load_pairs(files):
    imgs = np.empty((len(files), SIZE, SIZE, 3), np.uint8)
    msks = np.empty((len(files), SIZE, SIZE), np.uint8)
    for i, p in enumerate(files):
        im = Image.open(p).convert('RGB').resize((SIZE, SIZE), Image.BILINEAR)
        mp = p.replace('/Image/', '/GT_Object/').replace('.jpg', '.png')
        mk = Image.open(mp).convert('L').resize((SIZE, SIZE), Image.NEAREST)   # raises if mask missing
        imgs[i] = np.asarray(im); msks[i] = (np.asarray(mk) > 127)
        if i % 500 == 0:
            print(f'  caching {i}/{len(files)}', flush=True)
    return torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(msks).unsqueeze(1).contiguous()


def get_data():
    os.makedirs(CACHE, exist_ok=True)
    path = f'{CACHE}/cod10k_cam.pt'
    if os.path.exists(path):
        d = torch.load(path)
        print('loaded cache', {k: tuple(v[0].shape) for k, v in d.items()}, flush=True)
        return d
    files = sorted(glob.glob(f'{ROOT}/**/Image/COD10K-CAM-*.jpg', recursive=True))
    tr = [f for f in files if '/train/' in f.lower()]
    te = [f for f in files if '/test/' in f.lower()]
    print(f'camouflaged images found: train={len(tr)} test={len(te)} (expected 3040 / 2026)', flush=True)
    if len(tr) != 3040 or len(te) != 2026:
        raise RuntimeError(f'Unexpected counts. Sample path: {files[:1]}. Fix the glob/split logic.')
    d = {'train': load_pairs(tr), 'test': load_pairs(te)}
    torch.save(d, path)
    return d


# ----------------------------------------------------------------------------- simulation helpers
from sgn import simulate_lwir


def make_lwir(rgb, mask, delta):
    lw = simulate_lwir(rgb, mask, delta=delta)
    if lw.dim() == 3:
        lw = lw.unsqueeze(1)
    lw = lw.float()
    if lw.max() > 1.5:          # in case sgn returns 0..255
        lw = lw / 255.
    return lw.clamp(0, 1)


def shadow(rgb, p=0.3):
    B = rgb.size(0)
    eta = torch.randn(B, 1, 8, 8, device=rgb.device)
    m = 0.25 + 0.75 * torch.sigmoid(2 * F.interpolate(eta, size=rgb.shape[-2:], mode='bilinear', align_corners=False))
    sel = (torch.rand(B, 1, 1, 1, device=rgb.device) < p)
    return torch.where(sel, rgb * m, rgb)


def norm_rgb(x):
    return (x - MEAN.to(x.device)) / STD.to(x.device)


def norm_lw(lw):
    return norm_rgb(lw.repeat(1, 3, 1, 1))


# ----------------------------------------------------------------------------- raw-LWIR descriptors
def _rp(x, p):
    return F.pad(x, (p, p, p, p), mode='replicate')


def descriptors(lw):
    """lw: B,1,H,W raw thermal in [0,1]. Returns B,4 = log[local std, grad mean, high-pass energy, grad p99]."""
    x = lw.float()
    k = torch.ones(1, 1, 7, 7, device=x.device) / 49.
    m = F.conv2d(_rp(x, 3), k); v = (F.conv2d(_rp(x * x, 3), k) - m * m).clamp_min(1e-10)
    lstd = v.sqrt().mean((1, 2, 3))
    sx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]], device=x.device).view(1, 1, 3, 3)
    sy = sx.transpose(2, 3)
    xp = _rp(x, 1)
    g = torch.sqrt(F.conv2d(xp, sx) ** 2 + F.conv2d(xp, sy) ** 2 + 1e-10)
    gmean = g.mean((1, 2, 3))
    gp99 = torch.quantile(g.flatten(1), 0.99, dim=1)
    hp = (x - F.avg_pool2d(_rp(x, 2), 5, 1)).abs().mean((1, 2, 3))
    return torch.log(torch.stack([lstd, gmean, hp, gp99], 1) + 1e-5)


# ----------------------------------------------------------------------------- model
def feats(bb, x):
    x = bb.maxpool(bb.relu(bb.bn1(bb.conv1(x))))
    x = bb.layer1(x); f3 = bb.layer2(x); f4 = bb.layer3(f3); f5 = bb.layer4(f4)
    return [f3, f4, f5]


def backbone():
    w = None if os.environ.get('PHASE1_NOPRETRAIN') else tvm.ResNet18_Weights.DEFAULT
    bb = tvm.resnet18(weights=w)
    bb.fc = nn.Identity()
    return bb


class Net(nn.Module):
    def __init__(self, fusion):
        super().__init__()
        self.fusion = fusion
        chans = [128, 256, 512]
        self.rgb = backbone()
        if fusion != 'none':
            self.ir = backbone()
            self.mix = nn.ModuleList([nn.Sequential(nn.Conv2d(2 * c, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(True)) for c in chans])
        if fusion == 'gate':
            self.register_buffer('mu', torch.zeros(4)); self.register_buffer('sd', torch.ones(4))
            self.gate = nn.Sequential(nn.Linear(4, 32), nn.GELU(), nn.Linear(32, 3))
            nn.init.zeros_(self.gate[2].weight); nn.init.zeros_(self.gate[2].bias)   # alpha starts at 0.5
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in chans])
        self.head = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True))
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, rgb_n, lw_n=None, lw_raw=None):
        fr = feats(self.rgb, rgb_n)
        alpha = None
        if self.fusion != 'none':
            fi = feats(self.ir, lw_n)
            if self.fusion == 'gate':
                with torch.autocast(DEV, enabled=False):
                    z = (descriptors(lw_raw) - self.mu) / self.sd
                    alpha = torch.sigmoid(self.gate(z))          # B,3 (one thermal weight per stage)
            fr = [self.mix[i](torch.cat([fr[i], fi[i] * (1. if alpha is None else alpha[:, i].view(-1, 1, 1, 1).to(fi[i].dtype))], 1))
                  for i in range(3)]
        s = fr[0].shape[-2:]
        y = sum(F.interpolate(l(f), size=s, mode='bilinear', align_corners=False) for l, f in zip(self.lat, fr))
        y = self.out(self.head(y))
        return F.interpolate(y, size=rgb_n.shape[-2:], mode='bilinear', align_corners=False), alpha


def seg_loss(logit, m):
    bce = F.binary_cross_entropy_with_logits(logit, m)
    p = torch.sigmoid(logit)
    inter = (p * m).sum((1, 2, 3)); union = (p + m - p * m).sum((1, 2, 3))
    return bce + (1 - (inter + 1) / (union + 1)).mean()


def iou_per_image(logit, m):
    pred = (torch.sigmoid(logit.float()) > 0.5).float()
    inter = (pred * m).sum((1, 2, 3)); union = (pred + m - pred * m).sum((1, 2, 3))
    return (inter + 1e-6) / (union + 1e-6)


@torch.no_grad()
def calibrate_gate(net, tr_rgb, tr_msk, n=256):
    n = min(n, len(tr_rgb))
    idx = torch.randperm(len(tr_rgb), device=tr_rgb.device)[:n]
    zs = []
    DT = torch.tensor(DELTAS_TRAIN, device=tr_rgb.device)
    for i in range(0, n, 32):
        j = idx[i:i + 32]
        d = DT[torch.randint(0, len(DT), (len(j),), device=tr_rgb.device)]
        zs.append(descriptors(make_lwir(tr_rgb[j].float() / 255, tr_msk[j].float(), d)))
    z = torch.cat(zs)
    net.mu.copy_(z.mean(0)); net.sd.copy_(z.std(0) + 1e-3)


# ----------------------------------------------------------------------------- train / eval
def train_one(name, seed, data, args, out):
    cfg = CONFIGS[name]
    seed_all(seed)
    net = Net(cfg['fusion']).to(DEV)
    tr_rgb, tr_msk = data['train'][0].to(DEV), data['train'][1].to(DEV)
    if cfg['fusion'] == 'gate':
        calibrate_gate(net, tr_rgb, tr_msk)
    B, N = 16, len(tr_rgb)
    ipe = N // B
    opt = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=args.epochs * ipe)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    DT = torch.tensor(DELTAS_TRAIN, device=DEV)
    t0 = time.time(); step = 0
    for ep in range(args.epochs):
        net.train()
        perm = torch.randperm(N, device=DEV)
        run = 0.
        for it in range(ipe):
            idx = perm[it * B:(it + 1) * B]
            rgb = tr_rgb[idx].float() / 255; msk = tr_msk[idx].float()
            flip = (torch.rand(B, device=DEV) < 0.5).view(-1, 1, 1, 1)
            rgb = torch.where(flip, rgb.flip(-1), rgb); msk = torch.where(flip, msk.flip(-1), msk)
            delta = DT[torch.randint(0, len(DT), (B,), device=DEV)]
            drop = torch.rand(B, device=DEV) < cfg.get('drop', 0.0)
            delta_eff = torch.where(drop, torch.zeros_like(delta), delta)
            with torch.no_grad():
                lw = make_lwir(rgb, msk, delta_eff if cfg.get('mode') == 'd0' else delta) if cfg['fusion'] != 'none' else None
                if lw is not None and cfg.get('mode') == 'zero':
                    lw = lw * (~drop).float().view(-1, 1, 1, 1)
                rgb = shadow(rgb)                                   # shadow hits the visible channel only
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, alpha = net(norm_rgb(rgb), None if lw is None else norm_lw(lw), lw)
            loss = seg_loss(logit.float(), msk)
            if cfg.get('sup') and alpha is not None:
                tgt = torch.sigmoid(K_SIG * (delta_eff - D0_SIG)).view(-1, 1).expand_as(alpha)
                loss = loss + LAMBDA_SUP * F.mse_loss(alpha, tgt)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); step += 1
            if step % 50 == 0 or step == 1:
                el = time.time() - t0
                tot = args.epochs * ipe
                print(f'[{name} s{seed}] ep {ep + 1}/{args.epochs} it {it + 1}/{ipe} loss {run / (it + 1):.4f} '
                      f'{1000 * el / step:.0f} ms/it elapsed {el / 60:.1f}m ETA {(tot - step) * el / step / 60:.1f}m', flush=True)
    os.makedirs(f'{out}/ckpt', exist_ok=True)
    torch.save(net.state_dict(), f'{out}/ckpt/{name}_s{seed}.pt')
    res = evaluate(net, cfg, data['test'])
    res.update(cfg=name, seed=seed, train_min=round((time.time() - t0) / 60, 2))
    print(f'[{name} s{seed}] DONE mIoU', {k: round(v, 1) for k, v in res['miou'].items()},
          'alpha(P3)', None if res['alpha'] is None else {k: round(v[0], 2) for k, v in res['alpha'].items()}, flush=True)
    return res


@torch.no_grad()
def evaluate(net, cfg, te, bs=32):
    net.eval()
    te_rgb, te_msk = te
    miou, alph = {}, {}
    for d in DELTAS_TEST:
        seed_all(EVAL_SEED)                                         # identical thermal noise for every model
        ious, als = [], []
        for i in range(0, len(te_rgb), bs):
            rgb = te_rgb[i:i + bs].to(DEV).float() / 255; msk = te_msk[i:i + bs].to(DEV).float()
            lw = None if cfg['fusion'] == 'none' else make_lwir(rgb, msk, torch.full((len(rgb),), float(d), device=DEV))
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, alpha = net(norm_rgb(rgb), None if lw is None else norm_lw(lw), lw)
            ious.append(iou_per_image(logit, msk).cpu())
            if alpha is not None:
                als.append(alpha.float().cpu())
        miou[str(d)] = 100 * torch.cat(ious).mean().item()
        if als:
            alph[str(d)] = torch.cat(als).mean(0).tolist()
    return dict(miou=miou, alpha=alph or None)


def load_net(name, seed, out):
    net = Net(CONFIGS[name]['fusion']).to(DEV)
    net.load_state_dict(torch.load(f'{out}/ckpt/{name}_s{seed}.pt', map_location=DEV)); net.eval()
    return net


@torch.no_grad()
def run_rule(seed, data, out):
    a, b = load_net('rgb_only', seed, out), load_net('concat', seed, out)
    tr_rgb, tr_msk = data['train'][0].to(DEV), data['train'][1].to(DEV)
    seed_all(seed)
    sub = torch.randperm(len(tr_rgb), device=DEV)[:min(400, len(tr_rgb))]
    stats, labels = [], []
    for d in DELTAS_TRAIN:
        for i in range(0, len(sub), 50):
            j = sub[i:i + 50]
            lw = make_lwir(tr_rgb[j].float() / 255, tr_msk[j].float(), torch.full((len(j),), float(d), device=DEV))
            stats.append(descriptors(lw)[:, 3].cpu()); labels.append(torch.full((len(j),), float(d >= 1)))
    s = torch.cat(stats).numpy(); y = torch.cat(labels).numpy() > 0.5

    def bacc(t):
        p = s >= t
        return 0.5 * ((p & y).sum() / max(y.sum(), 1) + ((~p) & (~y)).sum() / max((~y).sum(), 1))
    tau = max(np.quantile(s, np.linspace(0.02, 0.98, 97)), key=bacc)
    print(f'[rule s{seed}] tau={tau:.3f} balanced-acc on train subset={bacc(tau):.3f}', flush=True)

    te_rgb, te_msk = data['test']
    R = {'rule': {}, 'oracle': {}}; frac = {}
    for d in DELTAS_TEST:
        seed_all(EVAL_SEED)
        ir, io, fr = [], [], []
        for i in range(0, len(te_rgb), 32):
            rgb = te_rgb[i:i + 32].to(DEV).float() / 255; msk = te_msk[i:i + 32].to(DEV).float()
            lw = make_lwir(rgb, msk, torch.full((len(rgb),), float(d), device=DEV))
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                la, _ = a(norm_rgb(rgb))
                lb, _ = b(norm_rgb(rgb), norm_lw(lw), lw)
            use_thermal = descriptors(lw)[:, 3] >= tau
            lr = torch.where(use_thermal.view(-1, 1, 1, 1), lb.float(), la.float())
            lo = lb.float() if d >= 1 else la.float()
            ir.append(iou_per_image(lr, msk).cpu()); io.append(iou_per_image(lo, msk).cpu())
            fr.append((~use_thermal).float().cpu())
        R['rule'][str(d)] = 100 * torch.cat(ir).mean().item()
        R['oracle'][str(d)] = 100 * torch.cat(io).mean().item()
        frac[str(d)] = torch.cat(fr).mean().item()
    print(f'[rule s{seed}] rule  ', {k: round(v, 1) for k, v in R['rule'].items()}, 'routed-to-RGB', {k: round(v, 2) for k, v in frac.items()})
    print(f'[rule s{seed}] oracle', {k: round(v, 1) for k, v in R['oracle'].items()}, flush=True)
    return [dict(cfg='rule', seed=seed, miou=R['rule'], alpha=None, tau=float(tau), routed_rgb=frac),
            dict(cfg='oracle', seed=seed, miou=R['oracle'], alpha=None)]


# ----------------------------------------------------------------------------- report
def report(out):
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    rows = [json.loads(l) for l in open(f'{out}/results.jsonl')]
    order = list(CONFIGS) + ['rule', 'oracle']
    cfgs = [c for c in order if any(r['cfg'] == c for r in rows)]
    print('mIoU % (mean +- std over seeds)   delta:', DELTAS_TEST)
    M = {}
    for c in cfgs:
        A = np.array([[r['miou'][str(d)] for d in DELTAS_TEST] for r in rows if r['cfg'] == c])
        M[c] = A
        print(f'{c:15s} n={len(A)} ' + '  '.join(f'{m:5.1f}+-{s:3.1f}' for m, s in zip(A.mean(0), A.std(0))))
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for c in cfgs:
        ax.errorbar(DELTAS_TEST, M[c].mean(0), yerr=M[c].std(0), marker='o', capsize=2, label=c, ls='--' if c == 'oracle' else '-')
    ax.invert_xaxis(); ax.set_xlabel('thermal contrast delta (K)'); ax.set_ylabel('mIoU (%)'); ax.legend(fontsize=7); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f'{out}/miou_vs_delta.png', dpi=200)
    gates = [c for c in cfgs if c.startswith('gate')]
    if gates:
        fig, axs = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
        for st, ax in enumerate(axs):
            for c in gates:
                A = np.array([[r['alpha'][str(d)][st] for d in DELTAS_TEST] for r in rows if r['cfg'] == c])
                ax.errorbar(DELTAS_TEST, A.mean(0), yerr=A.std(0), marker='o', capsize=2, label=c)
            ax.set_title(f'P{st + 3}'); ax.invert_xaxis(); ax.set_xlabel('delta (K)'); ax.grid(alpha=.3)
        axs[0].set_ylabel('mean thermal gate'); axs[0].legend(fontsize=7)
        fig.tight_layout(); fig.savefig(f'{out}/alpha_vs_delta.png', dpi=200)
    print('saved figures to', out)


# ----------------------------------------------------------------------------- main
if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--configs', default=','.join(CONFIGS))
    ap.add_argument('--seeds', default='0')
    ap.add_argument('--epochs', type=int, default=12)
    ap.add_argument('--smoke', action='store_true')
    ap.add_argument('--rule', action='store_true')
    ap.add_argument('--report', action='store_true')
    args = ap.parse_args()
    out = f'{BASE}/phase1' + ('_smoke' if args.smoke else '')
    os.makedirs(out, exist_ok=True)
    torch.backends.cudnn.benchmark = True
    if args.report:
        report(out); sys.exit()
    data = get_data()
    if args.smoke:
        args.epochs = 1
        data = {'train': (data['train'][0][:160], data['train'][1][:160]), 'test': (data['test'][0][:64], data['test'][1][:64])}
    seeds = [int(s) for s in args.seeds.split(',')]
    res_path = f'{out}/results.jsonl'

    def done():
        return {(r['cfg'], r['seed']) for r in map(json.loads, open(res_path))} if os.path.exists(res_path) else set()

    if args.rule:
        for s in seeds:
            if ('rule', s) in done():
                continue
            with open(res_path, 'a') as f:
                for r in run_rule(s, data, out):
                    f.write(json.dumps(r) + '\n')
        sys.exit()
    for s in seeds:
        for c in args.configs.split(','):
            if (c, s) in done():
                print(f'skip {c} s{s} (already done)'); continue
            r = train_one(c, s, data, args, out)
            with open(res_path, 'a') as f:
                f.write(json.dumps(r) + '\n')
            torch.cuda.empty_cache()

In [ ]:
   !python -u /kaggle/working/phase1.py --smoke
   !python -u /kaggle/working/phase1.py --smoke --rule

In [ ]:
import subprocess, os

def launch(gpu, configs, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = f"python -u /kaggle/working/phase1.py --configs {configs} --seeds 0,1,2 > {log} 2>&1"
    return subprocess.Popen(cmd, shell=True, env=env, start_new_session=True)

pA = launch(0, "rgb_only,concat,concat_drop,concat_drop_d0", "/kaggle/working/logA.txt")
pB = launch(1, "gate_raw,gate_sup,gate_sup_drop", "/kaggle/working/logB.txt")
print("launched", pA.pid, pB.pid)

In [ ]:
!tail -n 3 /kaggle/working/logA.txt
!tail -n 3 /kaggle/working/logB.txt
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

In [ ]:
!grep -h DONE /kaggle/working/logA.txt /kaggle/working/logB.txt > /kaggle/working/done_lines.txt
!cp /kaggle/working/phase1/results.jsonl /kaggle/working/results_backup.jsonl
!cat /kaggle/working/done_lines.txt

In [ ]:
!tail -n 3 /kaggle/working/logA.txt /kaggle/working/logB.txt

In [ ]:
!python -u /kaggle/working/phase1.py --rule --seeds 0,1,2
!python -u /kaggle/working/phase1.py --report

In [ ]:
import zipfile, glob

with zipfile.ZipFile('/kaggle/working/phase1_completed_results.zip', 'w') as z:
    for f in glob.glob('/kaggle/working/phase1/*') + glob.glob('/kaggle/working/*.txt') + glob.glob('/kaggle/working/*.png'):
        z.write(f)

print("Backup created: /kaggle/working/phase1_completed_results.zip")

In [ ]:
# 1. Patch phase1.py to add the two sweep configurations
p = '/kaggle/working/phase1.py'
s = open(p).read()
if 'concat_d0_p10' not in s:
    s = s.replace(
        "    'gate_raw':",
        "    'concat_d0_p10': dict(fusion='concat', drop=0.10, mode='d0'),\n    'concat_d0_p50': dict(fusion='concat', drop=0.50, mode='d0'),\n    'gate_raw':",
        1
    )
    open(p, 'w').write(s)
    print("phase1.py successfully patched with p=0.10 and p=0.50 configs.")

# 2. Launch both configs in parallel across GPUs
import subprocess, os

def launch(gpu, configs, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = f"python -u /kaggle/working/phase1.py --configs {configs} --seeds 0,1,2 > {log} 2>&1"
    return subprocess.Popen(cmd, shell=True, env=env, start_new_session=True)

pA = launch(0, "concat_d0_p10", "/kaggle/working/log_sweep_p10.txt")
pB = launch(1, "concat_d0_p50", "/kaggle/working/log_sweep_p50.txt")
print(f"Launched sweep on GPU 0 (PID {pA.pid}) and GPU 1 (PID {pB.pid}). Runtime: ~20 mins.")

In [ ]:
!echo "=== 1. Active phase1.py Processes ==="
!ps aux | grep "[p]hase1.py"

!echo ""
!echo "=== 2. GPU Memory & Process Allocation ==="
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

!echo ""
!echo "=== 3. Current Progress & Live ETA ==="
!tail -n 3 /kaggle/working/log_sweep_p10.txt /kaggle/working/log_sweep_p50.txt

In [ ]:
!tail -n 2 /kaggle/working/log_sweep_p10.txt /kaggle/working/log_sweep_p50.txt

In [ ]:
# 1. Update the final summary table and figures
!python -u /kaggle/working/phase1.py --report

# 2. Package all results, figures, and logs into a single archive
import zipfile, glob, os
from IPython.display import FileLink, display

zip_path = '/kaggle/working/phase1_final_results.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    # Include all files from phase1/
    for f in glob.glob('/kaggle/working/phase1/*'):
        z.write(f, os.path.join('phase1', os.path.basename(f)))
    # Include root logs, diagnostic plots, and jsonl files
    for f in glob.glob('/kaggle/working/*.txt') + glob.glob('/kaggle/working/*.jsonl') + glob.glob('/kaggle/working/*.png'):
        z.write(f, os.path.basename(f))

print("\n" + "="*50)
print("All artifacts packaged successfully!")
print("Click the link below to download to your Mac:")
print("="*50)
display(FileLink('phase1_final_results.zip'))

In [ ]:
p = '/kaggle/working/phase1.py'
s = open(p).read()

def sub(s, old, new):
    assert s.count(old) == 1, f"anchor problem: {old!r} found {s.count(old)} times"
    return s.replace(old, new)

if "gate_oracle" not in s:
    s = sub(s, "CONFIGS = {", "LAST_DELTA = {}\nCONFIGS = {")
    s = sub(s, "    'gate_raw':", "    'gate_oracle':    dict(fusion='oracle'),\n    'gate_raw':")
    s = sub(s, "lw = simulate_lwir(rgb, mask, delta=delta)",
               "LAST_DELTA['d'] = delta\n    lw = simulate_lwir(rgb, mask, delta=delta)")
    s = sub(s, "if self.fusion == 'gate':",
               "if self.fusion == 'oracle':\n                alpha = torch.sigmoid(K_SIG * (LAST_DELTA['d'].float() - D0_SIG)).view(-1, 1).expand(-1, 3)\n            if self.fusion == 'gate':")
    open(p, 'w').write(s)
    print("patched OK")
else:
    print("already patched")

In [ ]:
import subprocess, os

def launch(gpu, seeds, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = f"python -u /kaggle/working/phase1.py --configs gate_oracle --seeds {seeds} > {log} 2>&1"
    return subprocess.Popen(cmd, shell=True, env=env, start_new_session=True)

pA = launch(0, "0,1", "/kaggle/working/log_oracle_a.txt")
pB = launch(1, "2",   "/kaggle/working/log_oracle_b.txt")
print("launched", pA.pid, pB.pid)

In [ ]:
!ps aux | grep "[p]hase1.py"
!tail -n 5 /kaggle/working/log_oracle_a.txt /kaggle/working/log_oracle_b.txt

In [ ]:
!tail -n 3 /kaggle/working/log_oracle_a.txt /kaggle/working/log_oracle_b.txt

In [ ]:
# 1. Print all three oracle seeds
!grep -h DONE /kaggle/working/log_oracle_a.txt /kaggle/working/log_oracle_b.txt

# 2. Update the report table with gate_oracle included
!python -u /kaggle/working/phase1.py --report

In [ ]:
import zipfile, glob, os
from IPython.display import FileLink, display

zip_path = '/kaggle/working/phase1_with_oracle.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    for f in glob.glob('/kaggle/working/phase1/*'):
        z.write(f, os.path.join('phase1', os.path.basename(f)))
    for f in glob.glob('/kaggle/working/*.txt') + glob.glob('/kaggle/working/*.jsonl') + glob.glob('/kaggle/working/*.png'):
        z.write(f, os.path.basename(f))

print("Final archive created: phase1_with_oracle.zip")
display(FileLink('phase1_with_oracle.zip'))

In [ ]:
"""Recompute Table 1 of the BioAI-2026 paper (mean +- sample std over seeds, 'Avg' column,
and the Welch tests quoted in the text) from results_all.jsonl.   Usage: python make_table1.py results_all.jsonl"""
import json, sys
import numpy as np
from scipy import stats

D = ['8', '4', '2', '1', '0']
rows = [json.loads(l) for l in open(sys.argv[1] if len(sys.argv) > 1 else 'results_all.jsonl')]
NAMES = [('rgb_only', 'RGB only'), ('concat', 'Concat (early fusion)'),
         ('concat_d0_p10', '+ crossover dropout p=0.10'), ('concat_drop_d0', '+ crossover dropout p=0.25'),
         ('concat_d0_p50', '+ crossover dropout p=0.50'), ('concat_drop', '+ blank-thermal dropout p=0.25'),
         ('gate_raw', 'Gate, unsupervised'), ('gate_sup', 'Gate, supervised'),
         ('gate_sup_drop', 'Gate, supervised + blank dropout'), ('gate_oracle', 'Oracle gate (true delta)'),
         ('rule', 'Threshold rule'), ('oracle', 'Oracle switch (true delta)')]

def arr(cfg, d=None):
    r = sorted((x for x in rows if x['cfg'] == cfg), key=lambda x: x['seed'])
    assert len(r) == 3, (cfg, len(r))
    return np.array([[x['miou'][k] for k in D] for x in r])

print(f"{'Method':38s}" + ''.join(f'  d={d:>2s}      ' for d in D) + '  Avg')
for cfg, nm in NAMES:
    A = arr(cfg)
    print(f'{nm:38s}' + ''.join(f'{m:6.1f}+-{s:3.1f} ' for m, s in zip(A.mean(0), A.std(0, ddof=1))) + f'  {A.mean(0).mean():5.1f}')

def welch(c1, c2, j):
    return stats.ttest_ind(arr(c1)[:, j], arr(c2)[:, j], equal_var=False).pvalue
print('\nWelch p-values (two-sided):')
for a, b, j, lab in [('concat_d0_p50', 'rgb_only', 4, 'drop 0.50 vs RGB-only, d=0'),
                     ('gate_sup', 'concat', 4, 'sup. gate vs Concat, d=0'), ('gate_sup', 'concat', 3, 'sup. gate vs Concat, d=1'),
                     ('gate_oracle', 'concat', 4, 'oracle gate vs Concat, d=0'), ('gate_oracle', 'concat', 3, 'oracle gate vs Concat, d=1'),
                     ('gate_oracle', 'gate_sup', 3, 'oracle gate vs sup. gate, d=1'), ('gate_oracle', 'rgb_only', 4, 'oracle gate vs RGB-only, d=0')]:
    print(f'  {lab:34s} p = {welch(a, b, j):.3f}')

print('\nThreshold rule: share of test images routed to RGB-only (mean over seeds)')
R = np.array([[x['routed_rgb'][k] for k in D] for x in rows if x['cfg'] == 'rule'])
print('  ', dict(zip(D, np.round(100 * R.mean(0)).astype(int))))

print('\nMean thermal gate at stage P3 (mean over seeds):')
for cfg in ('gate_raw', 'gate_sup', 'gate_sup_drop', 'gate_oracle'):
    a = np.array([[x['alpha'][k][0] for k in D] for x in rows if x['cfg'] == cfg]).mean(0)
    print(f'  {cfg:14s}', np.round(a, 2))

## Update

``` Now, from here, further updates of BioAI Would work upon ```


``` For starting again, with the old results, run this code ```



```
import os
import sys

# 1. Search for sgn.py inside Kaggle workspace
target_dir = None
for root, dirs, files in os.walk('/kaggle'):
    if 'sgn.py' in files:
        target_dir = root
        break

# 2. If not found, clone your repository directly
# if target_dir is None:
print("sgn.py not found. Cloning repo from GitHub...")
!git clone https://github.com/atulsingh369/SpectralGate-Net.git
target_dir = '/kaggle/working/SpectralGate-Net'

# 3. Add to Python path
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

print(f"Loaded module directory: {target_dir}")
```


In [ ]:
import os
import sys

# 1. Search for sgn.py inside Kaggle workspace
target_dir = None
for root, dirs, files in os.walk('/kaggle'):
    if 'sgn.py' in files:
        target_dir = root
        break

# 2. If not found, clone your repository directly
# if target_dir is None:
print("sgn.py not found. Cloning repo from GitHub...")
!git clone https://github.com/atulsingh369/SpectralGate-Net.git
target_dir = '/kaggle/working/SpectralGate-Net'

# 3. Add to Python path
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

print(f"Loaded module directory: {target_dir}")

In [ ]:
!wc -l /kaggle/working/SpectralGate-Net/results/final_combined_results.jsonl
!head -n 2 /kaggle/working/SpectralGate-Net/results/final_combined_results.jsonl
!find /kaggle/input -maxdepth 4 -name "COD10K-v3"

!mkdir -p /kaggle/working/phase1_resnet18_orig
!cp /kaggle/working/SpectralGate-Net/results/final_combined_results.jsonl /kaggle/working/phase1_resnet18_orig/results.jsonl

In [ ]:
%%writefile /kaggle/working/phase1_rev.py
#!/usr/bin/env python
"""
Revision experiments extending phase1.py:
  1. --sim hard      : eroded/dilated mask, internal thermal texture, sigma in [1.5,3.5], 1-3px shift
  2. --backbone      : resnet18 | resnet50
  3. --diag_rgb      : zero-thermal inference diagnostic on trained fusion checkpoints (0 retraining)
  4. --real_root     : real RGB-T dataset (Camo-M3FD / MFNet) stratified by empirical thermal contrast
  5. --report        : sample std (ddof=1) + Welch t-test p-values vs concat & rgb_only
"""
import os, sys, glob, json, time, random, argparse
sys.path.insert(0, '/kaggle/working/SpectralGate-Net')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvm
from PIL import Image
from scipy import stats

from sgn import simulate_lwir, planck_band, smooth_noise, gauss_blur

ROOT = '/kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3'
BASE = '/kaggle/working'
CACHE = f'{BASE}/phase1_cache'
SIZE = 352
DELTAS_TEST = [8, 4, 2, 1, 0]
DELTAS_TRAIN = [0, 0.5, 1, 2, 4, 8]
EVAL_SEED = 12345
LAMBDA_SUP, K_SIG, D0_SIG = 1.0, 4.0, 0.5
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEV == 'cuda'
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

LAST_DELTA = {}
CONFIGS = {
    'rgb_only':       dict(fusion='none'),
    'concat':         dict(fusion='concat'),
    'concat_drop':    dict(fusion='concat', drop=0.25, mode='zero'),
    'concat_drop_d0': dict(fusion='concat', drop=0.25, mode='d0'),
    'concat_d0_p10':  dict(fusion='concat', drop=0.10, mode='d0'),
    'concat_d0_p50':  dict(fusion='concat', drop=0.50, mode='d0'),
    'gate_oracle':    dict(fusion='oracle'),
    'gate_raw':       dict(fusion='gate'),
    'gate_sup':       dict(fusion='gate', sup=True),
    'gate_sup_drop':  dict(fusion='gate', sup=True, drop=0.25, mode='zero'),
}

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ----------------------------------------------------------------------------- COD10K data
def load_pairs(files):
    imgs = np.empty((len(files), SIZE, SIZE, 3), np.uint8)
    msks = np.empty((len(files), SIZE, SIZE), np.uint8)
    for i, p in enumerate(files):
        im = Image.open(p).convert('RGB').resize((SIZE, SIZE), Image.BILINEAR)
        mp = p.replace('/Image/', '/GT_Object/').replace('.jpg', '.png')
        mk = Image.open(mp).convert('L').resize((SIZE, SIZE), Image.NEAREST)
        imgs[i] = np.asarray(im); msks[i] = (np.asarray(mk) > 127)
        if i % 500 == 0:
            print(f'  caching {i}/{len(files)}', flush=True)
    return torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(msks).unsqueeze(1).contiguous()

def get_data(root=ROOT):
    os.makedirs(CACHE, exist_ok=True)
    path = f'{CACHE}/cod10k_cam.pt'
    if os.path.exists(path):
        d = torch.load(path, weights_only=False)
        print('loaded cache', {k: tuple(v[0].shape) for k, v in d.items()}, flush=True)
        return d
    files = sorted(glob.glob(f'{root}/**/Image/COD10K-CAM-*.jpg', recursive=True))
    tr = [f for f in files if '/train/' in f.lower()]
    te = [f for f in files if '/test/' in f.lower()]
    print(f'camouflaged images found: train={len(tr)} test={len(te)}', flush=True)
    if len(tr) != 3040 or len(te) != 2026:
        raise RuntimeError(f'Unexpected counts ({len(tr)}, {len(te)}). Sample: {files[:1]}')
    d = {'train': load_pairs(tr), 'test': load_pairs(te)}
    torch.save(d, path)
    return d

# ----------------------------------------------------------------------------- Simulators (orig & hard)
@torch.no_grad()
def simulate_lwir_hard(rgb, mask, delta, T0=295.0, netd=0.05):
    """Harder thermal simulator: jittered boundary, internal heat texture, variable blur, 1-3px shift."""
    B, _, H, W = rgb.shape
    lum = rgb.mean(1, keepdim=True)
    T = T0 + 2.0 * smooth_noise(B, H, W, 12, rgb.device) + 2.0 * (lum - 0.5)
    tex = torch.clamp(0.85 + 0.25 * smooth_noise(B, H, W, 16, rgb.device), 0.4, 1.3)

    T_out = torch.empty_like(T)
    for b in range(B):
        mb = mask[b:b+1]
        r = int(torch.randint(-3, 4, (1,), device=rgb.device).item())
        if r > 0:
            mb = F.max_pool2d(mb, 2 * r + 1, stride=1, padding=r)
        elif r < 0:
            mb = 1.0 - F.max_pool2d(1.0 - mb, 2 * (-r) + 1, stride=1, padding=-r)
        tb = T[b:b+1] + delta[b].to(rgb.device) * mb * tex[b:b+1]
        sig = float(1.5 + 2.0 * torch.rand(1, device=rgb.device).item())  # [1.5, 3.5] px
        tb = gauss_blur(tb, sigma=sig, k=11)
        dx = int(torch.randint(-3, 4, (1,), device=rgb.device).item())
        dy = int(torch.randint(-3, 4, (1,), device=rgb.device).item())
        if dx != 0 or dy != 0:
            pad = 3
            tp = F.pad(tb, (pad, pad, pad, pad), mode='replicate')
            tb = tp[:, :, pad - dy:pad - dy + H, pad - dx:pad - dx + W]
        T_out[b:b+1] = tb

    T_out = T_out + netd * torch.randn(T_out.shape, device=T_out.device)
    lo = planck_band(torch.full((1, 1, 1, 1), 280.0, device=T_out.device))
    hi = planck_band(torch.full((1, 1, 1, 1), 320.0, device=T_out.device))
    g = ((planck_band(T_out) - lo) / (hi - lo)).clamp(0, 1).float()
    return torch.round(g * 255) / 255

def make_lwir(rgb, mask, delta, sim='orig'):
    LAST_DELTA['d'] = delta
    lw = simulate_lwir(rgb, mask, delta=delta) if sim == 'orig' else simulate_lwir_hard(rgb, mask, delta=delta)
    if lw.dim() == 3:
        lw = lw.unsqueeze(1)
    lw = lw.float()
    if lw.max() > 1.5:
        lw = lw / 255.
    return lw.clamp(0, 1)

def shadow(rgb, p=0.3):
    B = rgb.size(0)
    eta = torch.randn(B, 1, 8, 8, device=rgb.device)
    m = 0.25 + 0.75 * torch.sigmoid(2 * F.interpolate(eta, size=rgb.shape[-2:], mode='bilinear', align_corners=False))
    sel = (torch.rand(B, 1, 1, 1, device=rgb.device) < p)
    return torch.where(sel, rgb * m, rgb)

def norm_rgb(x): return (x - MEAN.to(x.device)) / STD.to(x.device)
def norm_lw(lw): return norm_rgb(lw.repeat(1, 3, 1, 1))

# ----------------------------------------------------------------------------- descriptors & models
def _rp(x, p): return F.pad(x, (p, p, p, p), mode='replicate')

def descriptors(lw):
    x = lw.float()
    k = torch.ones(1, 1, 7, 7, device=x.device) / 49.
    m = F.conv2d(_rp(x, 3), k); v = (F.conv2d(_rp(x * x, 3), k) - m * m).clamp_min(1e-10)
    lstd = v.sqrt().mean((1, 2, 3))
    sx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]], device=x.device).view(1, 1, 3, 3)
    sy = sx.transpose(2, 3)
    xp = _rp(x, 1)
    g = torch.sqrt(F.conv2d(xp, sx) ** 2 + F.conv2d(xp, sy) ** 2 + 1e-10)
    gmean = g.mean((1, 2, 3))
    gp99 = torch.quantile(g.flatten(1), 0.99, dim=1)
    hp = (x - F.avg_pool2d(_rp(x, 2), 5, 1)).abs().mean((1, 2, 3))
    return torch.log(torch.stack([lstd, gmean, hp, gp99], 1) + 1e-5)

def feats(bb, x):
    x = bb.maxpool(bb.relu(bb.bn1(bb.conv1(x))))
    x = bb.layer1(x); f3 = bb.layer2(x); f4 = bb.layer3(f3); f5 = bb.layer4(f4)
    return [f3, f4, f5]

def make_backbone(arch='resnet18'):
    nopre = bool(os.environ.get('PHASE1_NOPRETRAIN'))
    if arch == 'resnet18':
        bb = tvm.resnet18(weights=None if nopre else tvm.ResNet18_Weights.DEFAULT)
        chans = [128, 256, 512]
    elif arch == 'resnet50':
        bb = tvm.resnet50(weights=None if nopre else tvm.ResNet50_Weights.DEFAULT)
        chans = [512, 1024, 2048]
    else:
        raise ValueError(arch)
    bb.fc = nn.Identity()
    return bb, chans

class Net(nn.Module):
    def __init__(self, fusion, arch='resnet18'):
        super().__init__()
        self.fusion = fusion
        self.rgb, chans = make_backbone(arch)
        if fusion != 'none':
            self.ir, _ = make_backbone(arch)
            self.mix = nn.ModuleList([nn.Sequential(nn.Conv2d(2 * c, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(True)) for c in chans])
        if fusion == 'gate':
            self.register_buffer('mu', torch.zeros(4)); self.register_buffer('sd', torch.ones(4))
            self.gate = nn.Sequential(nn.Linear(4, 32), nn.GELU(), nn.Linear(32, 3))
            nn.init.zeros_(self.gate[2].weight); nn.init.zeros_(self.gate[2].bias)
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in chans])
        self.head = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True))
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, rgb_n, lw_n=None, lw_raw=None):
        fr = feats(self.rgb, rgb_n)
        alpha = None
        if self.fusion != 'none':
            fi = feats(self.ir, lw_n)
            if self.fusion == 'oracle':
                alpha = torch.sigmoid(K_SIG * (LAST_DELTA['d'].float() - D0_SIG)).view(-1, 1).expand(-1, 3)
            if self.fusion == 'gate':
                with torch.autocast(DEV, enabled=False):
                    z = (descriptors(lw_raw) - self.mu) / self.sd
                    alpha = torch.sigmoid(self.gate(z))
            fr = [self.mix[i](torch.cat([fr[i], fi[i] * (1. if alpha is None else alpha[:, i].view(-1, 1, 1, 1).to(fi[i].dtype))], 1))
                  for i in range(3)]
        s = fr[0].shape[-2:]
        y = sum(F.interpolate(l(f), size=s, mode='bilinear', align_corners=False) for l, f in zip(self.lat, fr))
        y = self.out(self.head(y))
        return F.interpolate(y, size=rgb_n.shape[-2:], mode='bilinear', align_corners=False), alpha

def seg_loss(logit, m):
    bce = F.binary_cross_entropy_with_logits(logit, m)
    p = torch.sigmoid(logit)
    inter = (p * m).sum((1, 2, 3)); union = (p + m - p * m).sum((1, 2, 3))
    return bce + (1 - (inter + 1) / (union + 1)).mean()

def iou_per_image(logit, m):
    pred = (torch.sigmoid(logit.float()) > 0.5).float()
    inter = (pred * m).sum((1, 2, 3)); union = (pred + m - pred * m).sum((1, 2, 3))
    return (inter + 1e-6) / (union + 1e-6)

@torch.no_grad()
def calibrate_gate(net, tr_rgb, tr_msk, sim='orig', n=256):
    n = min(n, len(tr_rgb))
    idx = torch.randperm(len(tr_rgb), device=tr_rgb.device)[:n]
    zs = []
    DT = torch.tensor(DELTAS_TRAIN, device=tr_rgb.device)
    for i in range(0, n, 32):
        j = idx[i:i + 32]
        d = DT[torch.randint(0, len(DT), (len(j),), device=tr_rgb.device)]
        zs.append(descriptors(make_lwir(tr_rgb[j].float() / 255, tr_msk[j].float(), d, sim=sim)))
    z = torch.cat(zs)
    net.mu.copy_(z.mean(0)); net.sd.copy_(z.std(0) + 1e-3)

# ----------------------------------------------------------------------------- COD10K train/eval/diag
def train_one(name, seed, data, args, out):
    cfg = CONFIGS[name]
    seed_all(seed)
    net = Net(cfg['fusion'], arch=args.backbone).to(DEV)
    tr_rgb, tr_msk = data['train'][0].to(DEV), data['train'][1].to(DEV)
    if cfg['fusion'] == 'gate':
        calibrate_gate(net, tr_rgb, tr_msk, sim=args.sim)
    B, N = 16, len(tr_rgb)
    ipe = N // B
    opt = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=args.epochs * ipe)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    DT = torch.tensor(DELTAS_TRAIN, device=DEV)
    t0 = time.time(); step = 0
    for ep in range(args.epochs):
        net.train(); perm = torch.randperm(N, device=DEV); run = 0.
        for it in range(ipe):
            idx = perm[it * B:(it + 1) * B]
            rgb = tr_rgb[idx].float() / 255; msk = tr_msk[idx].float()
            flip = (torch.rand(B, device=DEV) < 0.5).view(-1, 1, 1, 1)
            rgb = torch.where(flip, rgb.flip(-1), rgb); msk = torch.where(flip, msk.flip(-1), msk)
            delta = DT[torch.randint(0, len(DT), (B,), device=DEV)]
            drop = torch.rand(B, device=DEV) < cfg.get('drop', 0.0)
            delta_eff = torch.where(drop, torch.zeros_like(delta), delta)
            with torch.no_grad():
                lw = make_lwir(rgb, msk, delta_eff if cfg.get('mode') == 'd0' else delta, sim=args.sim) if cfg['fusion'] != 'none' else None
                if lw is not None and cfg.get('mode') == 'zero':
                    lw = lw * (~drop).float().view(-1, 1, 1, 1)
                rgb = shadow(rgb)
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, alpha = net(norm_rgb(rgb), None if lw is None else norm_lw(lw), lw)
            loss = seg_loss(logit.float(), msk)
            if cfg.get('sup') and alpha is not None:
                tgt = torch.sigmoid(K_SIG * (delta_eff - D0_SIG)).view(-1, 1).expand_as(alpha)
                loss = loss + LAMBDA_SUP * F.mse_loss(alpha, tgt)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); step += 1
            if step % 50 == 0 or step == 1:
                el = time.time() - t0; tot = args.epochs * ipe
                print(f'[{args.backbone}|{args.sim}|{name} s{seed}] ep {ep+1}/{args.epochs} it {it+1}/{ipe} '
                      f'loss {run/(it+1):.4f} {1000*el/step:.0f}ms/it {el/60:.1f}m ETA {(tot-step)*el/step/60:.1f}m', flush=True)
    os.makedirs(f'{out}/ckpt', exist_ok=True)
    torch.save(net.state_dict(), f'{out}/ckpt/{name}_s{seed}.pt')
    res = evaluate(net, cfg, data['test'], sim=args.sim)
    if cfg['fusion'] == 'concat':
        res['rgb_only_zero_th'] = eval_zero_thermal(net, data['test'])
    res.update(cfg=name, seed=seed, backbone=args.backbone, sim=args.sim, train_min=round((time.time()-t0)/60, 2))
    print(f'[{args.backbone}|{args.sim}|{name} s{seed}] DONE mIoU', {k: round(v, 1) for k, v in res['miou'].items()},
          'zero_th:', res.get('rgb_only_zero_th'), flush=True)
    return res

@torch.no_grad()
def evaluate(net, cfg, te, sim='orig', bs=32):
    net.eval(); te_rgb, te_msk = te; miou, alph = {}, {}
    for d in DELTAS_TEST:
        seed_all(EVAL_SEED); ious, als = [], []
        for i in range(0, len(te_rgb), bs):
            rgb = te_rgb[i:i+bs].to(DEV).float() / 255; msk = te_msk[i:i+bs].to(DEV).float()
            lw = None if cfg['fusion'] == 'none' else make_lwir(rgb, msk, torch.full((len(rgb),), float(d), device=DEV), sim=sim)
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, alpha = net(norm_rgb(rgb), None if lw is None else norm_lw(lw), lw)
            ious.append(iou_per_image(logit, msk).cpu())
            if alpha is not None: als.append(alpha.float().cpu())
        miou[str(d)] = 100 * torch.cat(ious).mean().item()
        if als: alph[str(d)] = torch.cat(als).mean(0).tolist()
    return dict(miou=miou, alpha=alph or None)

@torch.no_grad()
def eval_zero_thermal(net, te, bs=32):
    """Tests Section 7 hypothesis: evaluate trained fusion net with zeroed thermal channel."""
    net.eval(); te_rgb, te_msk = te; ious = []
    for i in range(0, len(te_rgb), bs):
        rgb = te_rgb[i:i+bs].to(DEV).float() / 255; msk = te_msk[i:i+bs].to(DEV).float()
        lw_z = torch.zeros(len(rgb), 1, rgb.shape[2], rgb.shape[3], device=DEV)
        with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
            logit, _ = net(norm_rgb(rgb), norm_lw(lw_z), lw_z)
        ious.append(iou_per_image(logit, msk).cpu())
    return round(100 * torch.cat(ious).mean().item(), 2)

@torch.no_grad()
def run_diag_rgb(seeds, data, out, arch):
    for c in ['concat', 'concat_drop_d0', 'concat_d0_p50']:
        vals = []
        for s in seeds:
            ckpt = f'{out}/ckpt/{c}_s{s}.pt'
            if not os.path.exists(ckpt): continue
            net = Net(CONFIGS[c]['fusion'], arch=arch).to(DEV)
            net.load_state_dict(torch.load(ckpt, map_location=DEV))
            vals.append(eval_zero_thermal(net, data['test']))
        if vals:
            a = np.array(vals)
            print(f'[diag zero-thermal | {c}] n={len(a)} mean={a.mean():.2f} +- {a.std(ddof=1) if len(a)>1 else 0:.2f} (vals={vals})')

# ----------------------------------------------------------------------------- Real RGB-T (Camo-M3FD / MFNet)
def load_real_rgbt(real_root):
    """Loads paired RGB, thermal, and binary mask images. Auto-splits 70/30 if no train/test dirs exist."""
    def find_triplets(base):
        def pick(cands):
            for c in cands:
                p = os.path.join(base, c)
                if os.path.isdir(p): return p
            return None
        rd = pick(['rgb', 'RGB', 'vi', 'VI', 'Image', 'images', 'visible'])
        td = pick(['thermal', 'Thermal', 'ir', 'IR', 'T', 'lwir'])
        md = pick(['mask', 'Mask', 'masks', 'GT', 'GT_Object', 'labels', 'SegmentationClass'])
        if not (rd and td and md): return []
        rmap = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{rd}/*'))}
        tmap = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{td}/*'))}
        mmap = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{md}/*'))}
        keys = sorted(set(rmap) & set(tmap) & set(mmap))
        return [(rmap[k], tmap[k], mmap[k]) for k in keys]

    tr_trips = find_triplets(os.path.join(real_root, 'train')) or find_triplets(os.path.join(real_root, 'Train'))
    te_trips = find_triplets(os.path.join(real_root, 'test')) or find_triplets(os.path.join(real_root, 'Test')) or find_triplets(os.path.join(real_root, 'val'))
    if not (tr_trips and te_trips):
        all_t = find_triplets(real_root)
        assert all_t, f'Could not find RGB/Thermal/Mask folders under {real_root}'
        rng = random.Random(42); rng.shuffle(all_t)
        cut = int(0.7 * len(all_t))
        tr_trips, te_trips = all_t[:cut], all_t[cut:]
    print(f'Real RGB-T loaded: train={len(tr_trips)} test={len(te_trips)}', flush=True)

    def pack(trips):
        N = len(trips)
        rgb = np.empty((N, SIZE, SIZE, 3), np.uint8)
        th  = np.empty((N, SIZE, SIZE), np.uint8)
        msk = np.empty((N, SIZE, SIZE), np.uint8)
        for i, (rp, tp, mp) in enumerate(trips):
            rgb[i] = np.asarray(Image.open(rp).convert('RGB').resize((SIZE, SIZE), Image.BILINEAR))
            th[i]  = np.asarray(Image.open(tp).convert('L').resize((SIZE, SIZE), Image.BILINEAR))
            ma = np.asarray(Image.open(mp).convert('L').resize((SIZE, SIZE), Image.NEAREST))
            msk[i] = (ma > 127) if ma.max() > 1 else (ma > 0)
        return (torch.from_numpy(rgb).permute(0, 3, 1, 2).contiguous(),
                torch.from_numpy(th).unsqueeze(1).contiguous(),
                torch.from_numpy(msk).unsqueeze(1).contiguous())
    return {'train': pack(tr_trips), 'test': pack(te_trips)}

def run_real_rgbt(args, out):
    data = load_real_rgbt(args.real_root)
    res_path = f'{out}/results.jsonl'
    done = {(r['cfg'], r['seed']) for r in map(json.loads, open(res_path))} if os.path.exists(res_path) else set()
    for s in [int(x) for x in args.seeds.split(',')]:
        for name in args.configs.split(','):
            if (name, s) in done: continue
            cfg = CONFIGS[name]; seed_all(s)
            net = Net(cfg['fusion'], arch=args.backbone).to(DEV)
            tr_r, tr_t, tr_m = [x.to(DEV) for x in data['train']]
            B, N = min(16, len(tr_r)), len(tr_r); ipe = max(1, N // B)
            opt = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=args.epochs * ipe)
            scaler = torch.amp.GradScaler('cuda', enabled=AMP)
            for ep in range(args.epochs):
                net.train(); perm = torch.randperm(N, device=DEV)
                for it in range(ipe):
                    idx = perm[it*B:(it+1)*B]
                    rgb = tr_r[idx].float() / 255; lw = tr_t[idx].float() / 255; msk = tr_m[idx].float()
                    flip = (torch.rand(len(idx), device=DEV) < 0.5).view(-1, 1, 1, 1)
                    rgb = torch.where(flip, rgb.flip(-1), rgb); lw = torch.where(flip, lw.flip(-1), lw); msk = torch.where(flip, msk.flip(-1), msk)
                    if cfg.get('drop', 0.0) > 0:
                        drop = (torch.rand(len(idx), device=DEV) < cfg['drop']).view(-1, 1, 1, 1)
                        # Replace dropped thermal with its spatial mean (contrast=0 proxy)
                        lw_mean = lw.mean((2, 3), keepdim=True).expand_as(lw)
                        lw = torch.where(drop, lw_mean, lw)
                    with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                        logit, _ = net(norm_rgb(rgb), None if cfg['fusion']=='none' else norm_lw(lw), lw)
                    loss = seg_loss(logit.float(), msk)
                    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(net.parameters(), 5.0); scaler.step(opt); scaler.update(); sched.step()
            # Eval stratified by empirical target-background contrast
            net.eval(); te_r, te_t, te_m = data['test']; ious, cons = [], []
            with torch.no_grad():
                for i in range(0, len(te_r), 32):
                    rgb = te_r[i:i+32].to(DEV).float()/255; lw = te_t[i:i+32].to(DEV).float()/255; msk = te_m[i:i+32].to(DEV).float()
                    with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                        logit, _ = net(norm_rgb(rgb), None if cfg['fusion']=='none' else norm_lw(lw), lw)
                    ious.append(iou_per_image(logit, msk).cpu())
                    dil = (F.max_pool2d(msk, 21, 1, 10) > 0.5) & (msk < 0.5)
                    for b in range(len(rgb)):
                        fg = msk[b] > 0.5; rg = dil[b]
                        cval = abs(lw[b][fg].mean().item() - lw[b][rg].mean().item()) if (fg.any() and rg.any()) else 0.0
                        cons.append(cval)
            ious = 100 * torch.cat(ious).numpy(); cons = np.array(cons)
            q33, q66 = np.quantile(cons, [0.33, 0.66])
            rec = dict(cfg=name, seed=s, overall=float(ious.mean()),
                       low_c=float(ious[cons <= q33].mean()),
                       mid_c=float(ious[(cons > q33) & (cons <= q66)].mean()),
                       high_c=float(ious[cons > q66].mean()),
                       q33=float(q33), q66=float(q66))
            print('[REAL]', rec, flush=True)
            with open(res_path, 'a') as f: f.write(json.dumps(rec) + '\n')

# ----------------------------------------------------------------------------- Report (ddof=1 + Welch p)
def report(out):
    rows = [json.loads(l) for l in open(f'{out}/results.jsonl')]
    if 'overall' in rows[0]:
        for c in dict.fromkeys(r['cfg'] for r in rows):
            rs = [r for r in rows if r['cfg'] == c]
            for k in ['overall', 'low_c', 'mid_c', 'high_c']:
                v = np.array([r[k] for r in rs])
                sd = v.std(ddof=1) if len(v) > 1 else 0.0
                print(f'{c:16s} n={len(v)} {k:7s}: {v.mean():.1f}+-{sd:.1f}')
        return
    order = list(CONFIGS) + ['rule', 'oracle']
    cfgs = [c for c in order if any(r['cfg'] == c for r in rows)]
    print('mIoU % (mean +- sample std, ddof=1)   delta:', DELTAS_TEST)
    M = {}
    for c in cfgs:
        A = np.array([[r['miou'][str(d)] for d in DELTAS_TEST] for r in rows if r['cfg'] == c])
        M[c] = A
        sd = A.std(0, ddof=1) if len(A) > 1 else np.zeros(len(DELTAS_TEST))
        print(f'{c:16s} n={len(A)} ' + '  '.join(f'{m:5.1f}+-{s:3.1f}' for m, s in zip(A.mean(0), sd)) + f'  | Avg={A.mean():.1f}')
    if 'concat' in M and len(M['concat']) > 1:
        print('\nWelch p-values at delta=0 vs concat:')
        for c in cfgs:
            if c != 'concat' and len(M[c]) > 1:
                _, p0 = stats.ttest_ind(M[c][:, -1], M['concat'][:, -1], equal_var=False)
                print(f'  {c:16s} vs concat (d=0): p={p0:.4g}')

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--configs', default='rgb_only,concat,concat_drop_d0,concat_d0_p50,gate_oracle')
    ap.add_argument('--seeds', default='0,1,2')
    ap.add_argument('--epochs', type=int, default=12)
    ap.add_argument('--backbone', default='resnet18', choices=['resnet18', 'resnet50'])
    ap.add_argument('--sim', default='orig', choices=['orig', 'hard'])
    ap.add_argument('--out_dir', default='')
    ap.add_argument('--real_root', default='')
    ap.add_argument('--diag_rgb', action='store_true')
    ap.add_argument('--report', action='store_true')
    ap.add_argument('--cache_only', action='store_true')
    args = ap.parse_args()

    out = args.out_dir or (f'{BASE}/phase1_real' if args.real_root else f'{BASE}/phase1_{args.backbone}_{args.sim}')
    os.makedirs(out, exist_ok=True)
    torch.backends.cudnn.benchmark = True

    if args.report:
        report(out); sys.exit()
    if args.real_root:
        run_real_rgbt(args, out); sys.exit()

    data = get_data()
    if args.cache_only: sys.exit()
    seeds = [int(s) for s in args.seeds.split(',')]
    if args.diag_rgb:
        run_diag_rgb(seeds, data, out, args.backbone); sys.exit()

    res_path = f'{out}/results.jsonl'
    def done():
        return {(r['cfg'], r['seed']) for r in map(json.loads, open(res_path))} if os.path.exists(res_path) else set()

    for s in seeds:
        for c in args.configs.split(','):
            if (c, s) in done():
                print(f'skip {c} s{s}'); continue
            r = train_one(c, s, data, args, out)
            with open(res_path, 'a') as f: f.write(json.dumps(r) + '\n')
            torch.cuda.empty_cache()

In [ ]:
!python -u /kaggle/working/phase1_rev.py --cache_only

## ResNet Loaded and Images too Cached

``` Now we woudl run the Harder Thermal Simulator ```

In [ ]:
import subprocess, os

def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/phase1_rev.py --sim hard --backbone resnet18 --configs rgb_only,concat,concat_drop_d0 --seeds 0,1,2", "/kaggle/working/log_hard_0.txt")
p1 = launch(1, "python -u /kaggle/working/phase1_rev.py --sim hard --backbone resnet18 --configs concat_d0_p50,gate_sup,gate_oracle --seeds 0,1,2", "/kaggle/working/log_hard_1.txt")
print("Hard-sim launched:", p0.pid, p1.pid)

In [ ]:
!ps aux | grep "[p]hase1_rev.py"
!tail -n 3 /kaggle/working/log_hard_0.txt /kaggle/working/log_hard_1.txt

In [ ]:
!python -u /kaggle/working/phase1_rev.py --sim hard --backbone resnet18 --report
!python -u /kaggle/working/phase1_rev.py --sim hard --backbone resnet18 --diag_rgb --seeds 0,1,2

### Saving Output for next session

In [ ]:
import zipfile, glob, os
from IPython.display import FileLink, display

with zipfile.ZipFile('/kaggle/working/checkpoint_stepC.zip', 'w') as z:
    for f in (glob.glob('/kaggle/working/phase1_*/results.jsonl')
              + glob.glob('/kaggle/working/log_*.txt')
              + ['/kaggle/working/phase1_rev.py']):
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, '/kaggle/working'))

display(FileLink('checkpoint_stepC.zip'))

In [ ]:
import subprocess, os

# Make sure cache exists (skips automatically if already present)
!python -u /kaggle/working/phase1_rev.py --cache_only

def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --configs rgb_only,concat,concat_drop_d0 --seeds 3,4", "/kaggle/working/log_s34_0.txt")
p1 = launch(1, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --configs concat_d0_p50,gate_sup,gate_oracle --seeds 3,4", "/kaggle/working/log_s34_1.txt")
print("5-seed extension launched:", p0.pid, p1.pid)

In [ ]:
# First verify your existing 3-seed table (from final_combined_results.jsonl)
!python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --report

# Now launch seeds 3 and 4 in parallel
p0 = launch(0, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --configs rgb_only,concat,concat_drop_d0 --seeds 3,4", "/kaggle/working/log_s34_0.txt")
p1 = launch(1, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --configs concat_d0_p50,gate_sup,gate_oracle --seeds 3,4", "/kaggle/working/log_s34_1.txt")
print("5-seed extension launched:", p0.pid, p1.pid)

In [ ]:
!ps aux | grep "[p]hase1_rev.py"
!tail -n 3 /kaggle/working/log_s34_0.txt /kaggle/working/log_s34_1.txt

In [ ]:
import json, subprocess, os

# 1. Repair results.jsonl using raw_decode (splits any glued }{ objects cleanly)
p = '/kaggle/working/phase1_resnet18_orig/results.jsonl'
raw = open(p).read()
dec = json.JSONDecoder()
rows, idx, n = [], 0, len(raw)
while idx < n:
    while idx < n and raw[idx].isspace():
        idx += 1
    if idx >= n:
        break
    obj, idx = dec.raw_decode(raw, idx)
    rows.append(obj)

seen = {(r['cfg'], r['seed']): r for r in rows}
with open(p, 'w') as f:
    for r in seen.values():
        f.write(json.dumps(r) + '\n')
print(f"Repaired results.jsonl! Valid unique runs: {len(seen)}")

# 2. Patch phase1_rev.py so future appends always start on a new line
code = open('/kaggle/working/phase1_rev.py').read()
old = "with open(res_path, 'a') as f: f.write(json.dumps(r) + '\\n')"
new = ("with open(res_path, 'a+') as f:\n"
       "                f.seek(0); txt = f.read()\n"
       "                if txt and not txt.endswith('\\n'): f.write('\\n')\n"
       "                f.write(json.dumps(r) + '\\n')")
if old in code:
    open('/kaggle/working/phase1_rev.py', 'w').write(code.replace(old, new, 1))
    print("Patched phase1_rev.py safely.")

# 3. Show current table (verifies recovered runs)
!python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --report

# 4. Relaunch remaining seeds 3,4 (already-completed ones will be skipped automatically)
def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --configs rgb_only,concat,concat_drop_d0 --seeds 3,4", "/kaggle/working/log_s34_0.txt")
p1 = launch(1, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --configs concat_d0_p50,gate_sup,gate_oracle --seeds 3,4", "/kaggle/working/log_s34_1.txt")
print("Relaunched cleanly:", p0.pid, p1.pid)

In [ ]:
import subprocess, os, zipfile, glob
from IPython.display import FileLink, display

# 1. Print 5-seed ResNet-18 report + zero-thermal diagnostic on orig
!python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --report
!python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --diag_rgb --seeds 3,4

# 2. Quick backup of everything finished so far
with zipfile.ZipFile('/kaggle/working/checkpoint_stepD.zip', 'w') as z:
    for f in glob.glob('/kaggle/working/phase1_*/results.jsonl') + glob.glob('/kaggle/working/log_*.txt'):
        z.write(f, os.path.relpath(f, '/kaggle/working'))
display(FileLink('checkpoint_stepD.zip'))

# 3. Launch Cell E: ResNet-50 (second architecture) in parallel on GPU 0 & GPU 1
def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet50 --configs rgb_only,concat --seeds 0,1,2", "/kaggle/working/log_r50_0.txt")
p1 = launch(1, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet50 --configs concat_drop_d0,gate_oracle --seeds 0,1,2", "/kaggle/working/log_r50_1.txt")
print("ResNet-50 launched:", p0.pid, p1.pid)

In [ ]:
p0 = launch(0, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet50 --configs rgb_only,concat --seeds 0,1,2", "/kaggle/working/log_r50_0.txt")
p1 = launch(1, "python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet50 --configs concat_drop_d0,gate_oracle --seeds 0,1,2", "/kaggle/working/log_r50_1.txt")
print("ResNet-50 launched:", p0.pid, p1.pid)

In [1]:
!ps aux | grep "[p]hase1_rev.py"
!tail -n 3 /kaggle/working/log_r50_0.txt /kaggle/working/log_r50_1.txt

==> /kaggle/working/log_r50_0.txt <==
[resnet50|orig|concat s2] ep 12/12 it 110/190 loss 0.3186 373ms/it 13.7m ETA 0.5m
[resnet50|orig|concat s2] ep 12/12 it 160/190 loss 0.3209 373ms/it 14.0m ETA 0.2m
[resnet50|orig|concat s2] DONE mIoU {'8': 88.1, '4': 87.9, '2': 87.1, '1': 81.3, '0': 52.4} zero_th: 0.08

==> /kaggle/working/log_r50_1.txt <==
[resnet50|orig|gate_oracle s2] ep 12/12 it 110/190 loss 0.3227 399ms/it 14.6m ETA 0.5m
[resnet50|orig|gate_oracle s2] ep 12/12 it 160/190 loss 0.3253 399ms/it 15.0m ETA 0.2m
[resnet50|orig|gate_oracle s2] DONE mIoU {'8': 88.1, '4': 88.0, '2': 87.4, '1': 82.5, '0': 53.2} zero_th: None


In [2]:
import zipfile, glob, os, urllib.request, json, re
from IPython.display import FileLink, display

print("=== 1. ResNet-50 Report ===")
!python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet50 --report

print("\n=== 2. ResNet-18 (5-seed) Report ===")
!python -u /kaggle/working/phase1_rev.py --sim orig --backbone resnet18 --report

# 3. Backup everything to Mac
with zipfile.ZipFile('/kaggle/working/checkpoint_stepE.zip', 'w') as z:
    for f in glob.glob('/kaggle/working/phase1_*/results.jsonl') + glob.glob('/kaggle/working/log_*.txt'):
        z.write(f, os.path.relpath(f, '/kaggle/working'))
display(FileLink('checkpoint_stepE.zip'))

# 4. Find Camo-M3FD / MFNet download URLs
print("\n=== 3. Checking arXiv 2604.16582 (Camo-M3FD) ===")
try:
    xml = urllib.request.urlopen("https://export.arxiv.org/api/query?id_list=2604.16582", timeout=10).read().decode()
    print("Title:", re.findall(r"<title>(.*?)</title>", xml, re.S))
    print("URLs:", re.findall(r"https?://[^\s<\"']+", xml))
except Exception as e:
    print("arXiv error:", e)

for q in ["Camo-M3FD", "MFNet+pytorch", "FMB+RGBT"]:
    print(f"\n=== GitHub search: {q} ===")
    try:
        req = urllib.request.Request(f"https://api.github.com/search/repositories?q={q}&per_page=5",
                                     headers={"User-Agent": "Mozilla/5.0"})
        data = json.loads(urllib.request.urlopen(req, timeout=10).read().decode())
        for item in data.get("items", []):
            print(" -", item["full_name"], "|", item["html_url"], "|", item["description"])
    except Exception as e:
        print("GitHub error:", e)

=== 1. ResNet-50 Report ===
mIoU % (mean +- sample std, ddof=1)   delta: [8, 4, 2, 1, 0]
rgb_only         n=4  56.5+-0.3   56.5+-0.3   56.5+-0.3   56.5+-0.3   56.5+-0.3  | Avg=56.5
concat           n=3  88.0+-0.5   87.8+-0.5   87.0+-0.5   81.3+-0.7   52.4+-0.1  | Avg=79.3
concat_drop_d0   n=3  87.7+-0.5   87.5+-0.5   86.6+-0.6   79.2+-0.6   55.0+-0.5  | Avg=79.2
gate_oracle      n=3  88.0+-0.5   87.9+-0.5   87.3+-0.5   82.3+-0.7   53.1+-0.1  | Avg=79.7

Welch p-values at delta=0 vs concat:
  rgb_only         vs concat (d=0): p=2.018e-05
  concat_drop_d0   vs concat (d=0): p=0.00894
  gate_oracle      vs concat (d=0): p=0.001476

=== 2. ResNet-18 (5-seed) Report ===
mIoU % (mean +- sample std, ddof=1)   delta: [8, 4, 2, 1, 0]
rgb_only         n=5  46.7+-0.3   46.7+-0.3   46.7+-0.3   46.7+-0.3   46.7+-0.3  | Avg=46.7
concat           n=5  87.8+-0.2   87.6+-0.2   86.7+-0.2   80.6+-0.4   42.2+-0.3  | Avg=77.0
concat_drop      n=3  87.4+-0.2   87.2+-0.2   86.1+-0.1   78.5+-0.2   44.7+-0.2  

/kaggle/working/checkpoint_stepE.zip


=== 3. Checking arXiv 2604.16582 (Camo-M3FD) ===
Title: ['arXiv Query: search_query=&amp;id_list=2604.16582&amp;start=0&amp;max_results=10', 'Camo-M3FD: A New Benchmark Dataset for Cross-Spectral Camouflaged Pedestrian Detection']
URLs: ['http://a9.com/-/spec/opensearch/1.1/', 'http://arxiv.org/schemas/atom', 'http://www.w3.org/2005/Atom', 'https://arxiv.org/api/7F7nNxhIV85480ov3Pb3MKaxq/M', 'https://arxiv.org/api/query?search_query=&amp;start=0&amp;max_results=10&amp;id_list=2604.16582', 'http://arxiv.org/abs/2604.16582v1', 'https://arxiv.org/abs/2604.16582v1', 'https://arxiv.org/pdf/2604.16582v1', 'https://cod-espol.github.io/Camo-M3FD/', 'http://arxiv.org/schemas/atom', 'http://arxiv.org/schemas/atom']

=== GitHub search: Camo-M3FD ===
 - COD-espol/Camo-M3FD | https://github.com/COD-espol/Camo-M3FD | None

=== GitHub search: MFNet+pytorch ===
 - cypw/PyTorch-MFNet | https://github.com/cypw/PyTorch-MFNet | None
 - haqishen/MFNet-pytorch | https://github.com/haqishen/MFNet-pytorch | 

### All Cells of reviewer comment ran, now checking Camo Dataset

In [3]:
!git clone https://github.com/COD-espol/Camo-M3FD.git /kaggle/working/Camo-M3FD
!find /kaggle/working/Camo-M3FD -maxdepth 3
!cat /kaggle/working/Camo-M3FD/README.md

import urllib.request, re
try:
    html = urllib.request.urlopen("https://cod-espol.github.io/Camo-M3FD/", timeout=10).read().decode("utf-8", "ignore")
    print("\n=== Project page download links ===")
    for u in sorted(set(re.findall(r'https?://[^\s"\'<>]+', html))):
        if any(k in u.lower() for k in ["drive", "mega", "zenodo", "huggingface", "dropbox", "onedrive", "zip", "tar", "rar", "box", "github"]):
            print(" -", u)
except Exception as e:
    print("Project page check error:", e)

Cloning into '/kaggle/working/Camo-M3FD'...
remote: Enumerating objects: 289, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 289 (delta 29), reused 30 (delta 4), pack-reused 209 (from 2)
Receiving objects: 100% (289/289), 61.55 MiB | 35.09 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/kaggle/working/Camo-M3FD
/kaggle/working/Camo-M3FD/.git
/kaggle/working/Camo-M3FD/.git/hooks
/kaggle/working/Camo-M3FD/.git/hooks/push-to-checkout.sample
/kaggle/working/Camo-M3FD/.git/hooks/pre-push.sample
/kaggle/working/Camo-M3FD/.git/hooks/prepare-commit-msg.sample
/kaggle/working/Camo-M3FD/.git/hooks/fsmonitor-watchman.sample
/kaggle/working/Camo-M3FD/.git/hooks/applypatch-msg.sample
/kaggle/working/Camo-M3FD/.git/hooks/pre-merge-commit.sample
/kaggle/working/Camo-M3FD/.git/hooks/update.sample
/kaggle/working/Camo-M3FD/.git/hooks/pre-commit.sample
/kaggle/working/Camo-M3FD/.git/hooks/pre-applypatch.sample
/kaggle/working/Camo-M

In [4]:
import re, urllib.request

# 1. Check all href links inside Camo-M3FD/index.html and sample image counts
html = open("/kaggle/working/Camo-M3FD/index.html", encoding="utf-8", errors="ignore").read()
print("=== Camo-M3FD index.html links ===")
for m in sorted(set(re.findall(r'href=["\']([^"\']+)["\']', html))):
    if not m.startswith("#") and not m.startswith("static/css") and not m.endswith(".ico"):
        print(" -", m)

!echo "\n=== Camo-M3FD static/images counts ==="
!ls /kaggle/working/Camo-M3FD/static/images/rgb 2>/dev/null | wc -l
!ls /kaggle/working/Camo-M3FD/static/images/thermal 2>/dev/null | wc -l
!ls /kaggle/working/Camo-M3FD/static/images/gt 2>/dev/null | wc -l

# 2. Also check MFNet-pytorch README for the direct MFNet dataset link (backup)
print("\n=== MFNet-pytorch README links ===")
mf = urllib.request.urlopen("https://raw.githubusercontent.com/haqishen/MFNet-pytorch/master/README.md", timeout=10).read().decode()
for line in mf.splitlines():
    if "http" in line or "dataset" in line.lower() or "download" in line.lower():
        print(" ", line)

=== Camo-M3FD index.html links ===
 - http://creativecommons.org/licenses/by-sa/4.0/
 - https://ajax.googleapis.com
 - https://cdn.jsdelivr.net
 - https://cdn.jsdelivr.net/gh/jpswalsh/academicons@1/css/academicons.min.css
 - https://cod-espol.github.io/Camo-M3FD/
 - https://documentcloud.adobe.com
 - https://fonts.googleapis.com
 - https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap
 - https://fonts.gstatic.com
 - https://github.com/eliahuhorwitz/Academic-project-page-template
 - https://nerfies.github.io
 - https://www.kaggle.com/datasets/hvelesaca/camo-m3fd
 - static/pdfs/camo_m3fd_paper.pdf
\n=== Camo-M3FD static/images counts ===
8
8
8

=== MFNet-pytorch README links ===
  Project home page: https://www.mi.t.u-tokyo.ac.jp/static/projects/mil_multispectral/
  This is official pytorch implementation of [MFNet: Towards real-time semantic segmentation for autonomous vehicles with multi-spectral scenes](https://ieeexplore.ieee.org/document/8206396/) (IRO

In [5]:
!find /kaggle/input -maxdepth 4 | grep -i camo

/kaggle/input/datasets/hvelesaca/camo-m3fd
/kaggle/input/datasets/hvelesaca/camo-m3fd/test
/kaggle/input/datasets/hvelesaca/camo-m3fd/train
/kaggle/input/datasets/hvelesaca/camo-m3fd/val


In [6]:
!find /kaggle/input/datasets/hvelesaca/camo-m3fd -maxdepth 3 -type d
!for d in /kaggle/input/datasets/hvelesaca/camo-m3fd/train/* /kaggle/input/datasets/hvelesaca/camo-m3fd/test/* /kaggle/input/datasets/hvelesaca/camo-m3fd/val/*; do echo "$d: $(ls -1 "$d" | wc -l) files | sample: $(ls -1 "$d" | head -n 2 | tr '\n' ' ')"; done

/kaggle/input/datasets/hvelesaca/camo-m3fd
/kaggle/input/datasets/hvelesaca/camo-m3fd/test
/kaggle/input/datasets/hvelesaca/camo-m3fd/test/GT
/kaggle/input/datasets/hvelesaca/camo-m3fd/test/Imgs
/kaggle/input/datasets/hvelesaca/camo-m3fd/test/Thermal
/kaggle/input/datasets/hvelesaca/camo-m3fd/train
/kaggle/input/datasets/hvelesaca/camo-m3fd/train/Canny
/kaggle/input/datasets/hvelesaca/camo-m3fd/train/Edge
/kaggle/input/datasets/hvelesaca/camo-m3fd/train/GT
/kaggle/input/datasets/hvelesaca/camo-m3fd/train/Gradient-Foreground
/kaggle/input/datasets/hvelesaca/camo-m3fd/train/Imgs
/kaggle/input/datasets/hvelesaca/camo-m3fd/train/Thermal
/kaggle/input/datasets/hvelesaca/camo-m3fd/val
/kaggle/input/datasets/hvelesaca/camo-m3fd/val/Canny
/kaggle/input/datasets/hvelesaca/camo-m3fd/val/Edge
/kaggle/input/datasets/hvelesaca/camo-m3fd/val/GT
/kaggle/input/datasets/hvelesaca/camo-m3fd/val/Gradient-Foreground
/kaggle/input/datasets/hvelesaca/camo-m3fd/val/Imgs
/kaggle/input/datasets/hvelesaca/camo-

## Got the data, Now running it resolve in our model

In [7]:
%%writefile /kaggle/working/camo_m3fd_eval.py
import os, sys, glob, json, time, random, argparse
sys.path.insert(0, '/kaggle/working')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from scipy import stats

from phase1_rev import Net, seg_loss, iou_per_image, norm_rgb, norm_lw, descriptors, seed_all, DEV, AMP, SIZE

CAMO_ROOT = '/kaggle/input/datasets/hvelesaca/camo-m3fd'
OUT_DIR = '/kaggle/working/phase1_camo_m3fd'

CONFIGS = {
    'rgb_only':      dict(fusion='none'),
    'concat':        dict(fusion='concat'),
    'concat_drop':   dict(fusion='concat', drop=0.25, mode='zero'),
    'concat_bg_p25': dict(fusion='concat', drop=0.25, mode='shuffle'),
    'concat_bg_p50': dict(fusion='concat', drop=0.50, mode='shuffle'),
    'gate_raw':      dict(fusion='gate'),
}

def load_split(split):
    base = os.path.join(CAMO_ROOT, split)
    im_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Imgs/*'))}
    th_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Thermal/*'))}
    gt_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/GT/*'))}
    keys = sorted(set(im_map) & set(th_map) & set(gt_map))
    N = len(keys)
    assert N > 0, f'No matched files in {base}'
    rgb = np.empty((N, SIZE, SIZE, 3), np.uint8)
    th  = np.empty((N, SIZE, SIZE), np.uint8)
    msk = np.empty((N, SIZE, SIZE), np.uint8)
    for i, k in enumerate(keys):
        rgb[i] = np.asarray(Image.open(im_map[k]).convert('RGB').resize((SIZE, SIZE), Image.BILINEAR))
        th[i]  = np.asarray(Image.open(th_map[k]).convert('L').resize((SIZE, SIZE), Image.BILINEAR))
        ma = np.asarray(Image.open(gt_map[k]).convert('L').resize((SIZE, SIZE), Image.NEAREST))
        msk[i] = (ma > 127) if ma.max() > 1 else (ma > 0)
    return (torch.from_numpy(rgb).permute(0, 3, 1, 2).contiguous(),
            torch.from_numpy(th).unsqueeze(1).contiguous(),
            torch.from_numpy(msk).unsqueeze(1).contiguous())

@torch.no_grad()
def eval_split(net, cfg, data_tuple, bs=32):
    net.eval()
    te_r, te_t, te_m = data_tuple
    ious, cons, ious_shuf, alphas = [], [], [], []
    N = len(te_r)
    # Deterministic mismatched-thermal permutation for real-background crossover check
    perm = torch.roll(torch.arange(N), shifts=max(1, N // 3))
    shuf_t = te_t[perm]

    for i in range(0, N, bs):
        rgb = te_r[i:i+bs].to(DEV).float() / 255
        lw  = te_t[i:i+bs].to(DEV).float() / 255
        lws = shuf_t[i:i+bs].to(DEV).float() / 255
        msk = te_m[i:i+bs].to(DEV).float()
        with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
            logit, alpha = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            logit_s, _   = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lws), lws)
        ious.append(iou_per_image(logit, msk).cpu())
        ious_shuf.append(iou_per_image(logit_s, msk).cpu())
        if alpha is not None:
            alphas.append(alpha.float().cpu())
        dil = (F.max_pool2d(msk, 21, 1, 10) > 0.5) & (msk < 0.5)
        for b in range(len(rgb)):
            fg = msk[b] > 0.5; rg = dil[b]
            c = abs(lw[b][fg].mean().item() - lw[b][rg].mean().item()) * 255.0 if (fg.any() and rg.any()) else 0.0
            cons.append(c)

    ious = 100 * torch.cat(ious).numpy()
    ious_shuf = 100 * torch.cat(ious_shuf).numpy()
    cons = np.array(cons)
    q33, q66 = np.quantile(cons, [0.33, 0.66])
    out = dict(
        overall=float(ious.mean()),
        low_c=float(ious[cons <= q33].mean()),
        mid_c=float(ious[(cons > q33) & (cons <= q66)].mean()),
        high_c=float(ious[cons > q66].mean()),
        crossover_realbg=float(ious_shuf.mean()),
        q33_gl=float(q33), q66_gl=float(q66),
        c_min=float(cons.min()), c_med=float(np.median(cons)), c_max=float(cons.max()),
    )
    if alphas:
        al = torch.cat(alphas).numpy()
        out['alpha_low_p3'] = float(al[cons <= q33, 0].mean())
        out['alpha_high_p3'] = float(al[cons > q66, 0].mean())
    return out

def train_and_eval(name, seed, tr, te, vt, epochs=20):
    cfg = CONFIGS[name]; seed_all(seed)
    net = Net(cfg['fusion'], arch='resnet18').to(DEV)
    tr_r, tr_t, tr_m = [x.to(DEV) for x in tr]
    if cfg['fusion'] == 'gate':
        with torch.no_grad():
            z = descriptors(tr_t[:256].float() / 255)
            net.mu.copy_(z.mean(0)); net.sd.copy_(z.std(0) + 1e-3)
    B, N = 16, len(tr_r); ipe = N // B
    opt = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=epochs * ipe)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    t0 = time.time()
    for ep in range(epochs):
        net.train(); perm = torch.randperm(N, device=DEV)
        for it in range(ipe):
            idx = perm[it*B:(it+1)*B]
            rgb = tr_r[idx].float() / 255; lw = tr_t[idx].float() / 255; msk = tr_m[idx].float()
            flip = (torch.rand(B, device=DEV) < 0.5).view(-1, 1, 1, 1)
            rgb = torch.where(flip, rgb.flip(-1), rgb); lw = torch.where(flip, lw.flip(-1), lw); msk = torch.where(flip, msk.flip(-1), msk)
            if cfg.get('drop', 0.0) > 0:
                drop = (torch.rand(B, device=DEV) < cfg['drop']).view(-1, 1, 1, 1)
                if cfg.get('mode') == 'zero':
                    lw = torch.where(drop, torch.zeros_like(lw), lw)
                else:
                    lw_shuf = torch.roll(lw, shifts=1, dims=0)
                    lw = torch.where(drop, lw_shuf, lw)
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, _ = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            loss = seg_loss(logit.float(), msk)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 5.0); scaler.step(opt); scaler.update(); sched.step()

    res = dict(cfg=name, seed=seed, min=round((time.time()-t0)/60, 2),
               test=eval_split(net, cfg, te), valtest=eval_split(net, cfg, vt))
    print(f'[Camo-M3FD | {name:14s} s{seed}] test={res["test"]["overall"]:.1f} '
          f'(low={res["test"]["low_c"]:.1f}, high={res["test"]["high_c"]:.1f}, xover={res["test"]["crossover_realbg"]:.1f})', flush=True)
    return res

def report():
    rows = [json.loads(l) for l in open(f'{OUT_DIR}/results.jsonl')]
    for split_key in ['test', 'valtest']:
        print(f'\n=== Camo-M3FD ({split_key}) | Contrast gray-level q33={rows[0][split_key]["q33_gl"]:.1f}, q66={rows[0][split_key]["q66_gl"]:.1f} (min={rows[0][split_key]["c_min"]:.1f}, max={rows[0][split_key]["c_max"]:.1f}) ===')
        M = {}
        for c in [k for k in CONFIGS if any(r['cfg'] == k for r in rows)]:
            rs = [r[split_key] for r in rows if r['cfg'] == c]
            M[c] = rs
            parts = []
            for m in ['overall', 'low_c', 'mid_c', 'high_c', 'crossover_realbg']:
                v = np.array([x[m] for x in rs])
                sd = v.std(ddof=1) if len(v) > 1 else 0.0
                parts.append(f'{m}={v.mean():5.1f}+-{sd:3.1f}')
            print(f'{c:15s} n={len(rs)}  ' + '  '.join(parts))
        if 'concat' in M and len(M['concat']) > 1:
            c_x = [x['crossover_realbg'] for x in M['concat']]
            c_l = [x['low_c'] for x in M['concat']]
            print('Welch p vs concat (low_c | crossover_realbg):')
            for c, rs in M.items():
                if c != 'concat' and len(rs) > 1:
                    _, pl = stats.ttest_ind([x['low_c'] for x in rs], c_l, equal_var=False)
                    _, px = stats.ttest_ind([x['crossover_realbg'] for x in rs], c_x, equal_var=False)
                    print(f'  {c:15s}: p_low={pl:.4g}, p_xover={px:.4g}')

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--configs', default=','.join(CONFIGS))
    ap.add_argument('--seeds', default='0,1,2,3,4')

Writing /kaggle/working/camo_m3fd_eval.py


In [14]:
import subprocess, os

def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4", "/kaggle/working/log_camo_0.txt")
p1 = launch(1, "python -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4", "/kaggle/working/log_camo_1.txt")
print("Camo-M3FD launched:", p0.pid, p1.pid)

Camo-M3FD launched: 341 342


In [17]:
!ps aux | grep "[c]amo_m3fd_eval.py"
!tail -n 5 /kaggle/working/log_camo_0.txt /kaggle/working/log_camo_1.txt

==> /kaggle/working/log_camo_0.txt <==

==> /kaggle/working/log_camo_1.txt <==


In [12]:
!python -u /kaggle/working/camo_m3fd_eval.py --report

import zipfile, glob, os
from IPython.display import FileLink, display
with zipfile.ZipFile('/kaggle/working/bioai_ALL_final.zip', 'w') as z:
    for f in (glob.glob('/kaggle/working/phase1_*/results.jsonl')
              + glob.glob('/kaggle/working/log_*.txt')
              + ['/kaggle/working/phase1_rev.py', '/kaggle/working/camo_m3fd_eval.py']):
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, '/kaggle/working'))
display(FileLink('bioai_ALL_final.zip'))

/kaggle/working/bioai_ALL_final.zip

In [18]:
%%writefile /kaggle/working/camo_m3fd_eval.py
import os, sys, glob, json, time, random, argparse
sys.path.insert(0, '/kaggle/working')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from scipy import stats

from phase1_rev import Net, seg_loss, iou_per_image, norm_rgb, norm_lw, descriptors, seed_all, DEV, AMP, SIZE

CAMO_ROOT = '/kaggle/input/datasets/hvelesaca/camo-m3fd'
OUT_DIR = '/kaggle/working/phase1_camo_m3fd'

CONFIGS = {
    'rgb_only':      dict(fusion='none'),
    'concat':        dict(fusion='concat'),
    'concat_drop':   dict(fusion='concat', drop=0.25, mode='zero'),
    'concat_bg_p25': dict(fusion='concat', drop=0.25, mode='shuffle'),
    'concat_bg_p50': dict(fusion='concat', drop=0.50, mode='shuffle'),
    'gate_raw':      dict(fusion='gate'),
}

def load_split(split):
    base = os.path.join(CAMO_ROOT, split)
    im_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Imgs/*'))}
    th_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Thermal/*'))}
    gt_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/GT/*'))}
    keys = sorted(set(im_map) & set(th_map) & set(gt_map))
    N = len(keys)
    assert N > 0, f'No matched files in {base}'
    rgb = np.empty((N, SIZE, SIZE, 3), np.uint8)
    th  = np.empty((N, SIZE, SIZE), np.uint8)
    msk = np.empty((N, SIZE, SIZE), np.uint8)
    for i, k in enumerate(keys):
        rgb[i] = np.asarray(Image.open(im_map[k]).convert('RGB').resize((SIZE, SIZE), Image.BILINEAR))
        th[i]  = np.asarray(Image.open(th_map[k]).convert('L').resize((SIZE, SIZE), Image.BILINEAR))
        ma = np.asarray(Image.open(gt_map[k]).convert('L').resize((SIZE, SIZE), Image.NEAREST))
        msk[i] = (ma > 127) if ma.max() > 1 else (ma > 0)
    return (torch.from_numpy(rgb).permute(0, 3, 1, 2).contiguous(),
            torch.from_numpy(th).unsqueeze(1).contiguous(),
            torch.from_numpy(msk).unsqueeze(1).contiguous())

@torch.no_grad()
def eval_split(net, cfg, data_tuple, bs=32):
    net.eval()
    te_r, te_t, te_m = data_tuple
    ious, cons, ious_shuf, alphas = [], [], [], []
    N = len(te_r)
    perm = torch.roll(torch.arange(N), shifts=max(1, N // 3))
    shuf_t = te_t[perm]

    for i in range(0, N, bs):
        rgb = te_r[i:i+bs].to(DEV).float() / 255
        lw  = te_t[i:i+bs].to(DEV).float() / 255
        lws = shuf_t[i:i+bs].to(DEV).float() / 255
        msk = te_m[i:i+bs].to(DEV).float()
        with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
            logit, alpha = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            logit_s, _   = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lws), lws)
        ious.append(iou_per_image(logit, msk).cpu())
        ious_shuf.append(iou_per_image(logit_s, msk).cpu())
        if alpha is not None:
            alphas.append(alpha.float().cpu())
        dil = (F.max_pool2d(msk, 21, 1, 10) > 0.5) & (msk < 0.5)
        for b in range(len(rgb)):
            fg = msk[b] > 0.5; rg = dil[b]
            c = abs(lw[b][fg].mean().item() - lw[b][rg].mean().item()) * 255.0 if (fg.any() and rg.any()) else 0.0
            cons.append(c)

    ious = 100 * torch.cat(ious).numpy()
    ious_shuf = 100 * torch.cat(ious_shuf).numpy()
    cons = np.array(cons)
    q33, q66 = np.quantile(cons, [0.33, 0.66])
    out = dict(
        overall=float(ious.mean()),
        low_c=float(ious[cons <= q33].mean()),
        mid_c=float(ious[(cons > q33) & (cons <= q66)].mean()),
        high_c=float(ious[cons > q66].mean()),
        crossover_realbg=float(ious_shuf.mean()),
        q33_gl=float(q33), q66_gl=float(q66),
        c_min=float(cons.min()), c_med=float(np.median(cons)), c_max=float(cons.max()),
    )
    if alphas:
        al = torch.cat(alphas).numpy()
        out['alpha_low_p3'] = float(al[cons <= q33, 0].mean())
        out['alpha_high_p3'] = float(al[cons > q66, 0].mean())
    return out

def train_and_eval(name, seed, tr, te, vt, epochs=20):
    cfg = CONFIGS[name]; seed_all(seed)
    net = Net(cfg['fusion'], arch='resnet18').to(DEV)
    tr_r, tr_t, tr_m = [x.to(DEV) for x in tr]
    if cfg['fusion'] == 'gate':
        with torch.no_grad():
            z = descriptors(tr_t[:256].float() / 255)
            net.mu.copy_(z.mean(0)); net.sd.copy_(z.std(0) + 1e-3)
    B, N = 16, len(tr_r); ipe = N // B
    opt = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=epochs * ipe)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    t0 = time.time()
    for ep in range(epochs):
        net.train(); perm = torch.randperm(N, device=DEV)
        for it in range(ipe):
            idx = perm[it*B:(it+1)*B]
            rgb = tr_r[idx].float() / 255; lw = tr_t[idx].float() / 255; msk = tr_m[idx].float()
            flip = (torch.rand(B, device=DEV) < 0.5).view(-1, 1, 1, 1)
            rgb = torch.where(flip, rgb.flip(-1), rgb); lw = torch.where(flip, lw.flip(-1), lw); msk = torch.where(flip, msk.flip(-1), msk)
            if cfg.get('drop', 0.0) > 0:
                drop = (torch.rand(B, device=DEV) < cfg['drop']).view(-1, 1, 1, 1)
                if cfg.get('mode') == 'zero':
                    lw = torch.where(drop, torch.zeros_like(lw), lw)
                else:
                    lw_shuf = torch.roll(lw, shifts=1, dims=0)
                    lw = torch.where(drop, lw_shuf, lw)
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, _ = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            loss = seg_loss(logit.float(), msk)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 5.0); scaler.step(opt); scaler.update(); sched.step()

    res = dict(cfg=name, seed=seed, min=round((time.time()-t0)/60, 2),
               test=eval_split(net, cfg, te), valtest=eval_split(net, cfg, vt))
    print(f'[Camo-M3FD | {name:14s} s{seed}] test={res["test"]["overall"]:.1f} '
          f'(low={res["test"]["low_c"]:.1f}, high={res["test"]["high_c"]:.1f}, xover={res["test"]["crossover_realbg"]:.1f})', flush=True)
    return res

def report():
    rows = [json.loads(l) for l in open(f'{OUT_DIR}/results.jsonl')]
    for split_key in ['test', 'valtest']:
        print(f'\n=== Camo-M3FD ({split_key}) | Contrast gray-level q33={rows[0][split_key]["q33_gl"]:.1f}, q66={rows[0][split_key]["q66_gl"]:.1f} (min={rows[0][split_key]["c_min"]:.1f}, max={rows[0][split_key]["c_max"]:.1f}) ===')
        M = {}
        for c in [k for k in CONFIGS if any(r['cfg'] == k for r in rows)]:
            rs = [r[split_key] for r in rows if r['cfg'] == c]
            M[c] = rs
            parts = []
            for m in ['overall', 'low_c', 'mid_c', 'high_c', 'crossover_realbg']:
                v = np.array([x[m] for x in rs])
                sd = v.std(ddof=1) if len(v) > 1 else 0.0
                parts.append(f'{m}={v.mean():5.1f}+-{sd:3.1f}')
            print(f'{c:15s} n={len(rs)}  ' + '  '.join(parts))
        if 'concat' in M and len(M['concat']) > 1:
            c_x = [x['crossover_realbg'] for x in M['concat']]
            c_l = [x['low_c'] for x in M['concat']]
            print('Welch p vs concat (low_c | crossover_realbg):')
            for c, rs in M.items():
                if c != 'concat' and len(rs) > 1:
                    _, pl = stats.ttest_ind([x['low_c'] for x in rs], c_l, equal_var=False)
                    _, px = stats.ttest_ind([x['crossover_realbg'] for x in rs], c_x, equal_var=False)
                    print(f'  {c:15s}: p_low={pl:.4g}, p_xover={px:.4g}')

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--configs', default=','.join(CONFIGS))
    ap.add_argument('--seeds', default='0,1,2,3,4')
    ap.add_argument('--epochs', type=int, default=20)
    ap.add_argument('--report', action='store_true')
    args = ap.parse_args()
    os.makedirs(OUT_DIR, exist_ok=True)
    if args.report:
        report(); sys.exit()
    tr, va, te = load_split('train'), load_split('val'), load_split('test')
    vt = tuple(torch.cat([va[i], te[i]], dim=0) for i in range(3))
    print(f'Loaded Camo-M3FD: train={len(tr[0])}, val={len(va[0])}, test={len(te[0])}', flush=True)
    res_path = f'{OUT_DIR}/results.jsonl'
    done = {(r['cfg'], r['seed']) for r in map(json.loads, open(res_path))} if os.path.exists(res_path) else set()
    for s in [int(x) for x in args.seeds.split(',')]:
        for c in args.configs.split(','):
            if (c, s) in done: continue
            r = train_and_eval(c, s, tr, te, vt, epochs=args.epochs)
            with open(res_path, 'a+') as f:
                f.seek(0); txt = f.read()
                if txt and not txt.endswith('\n'): f.write('\n')
                f.write(json.dumps(r) + '\n')
            torch.cuda.empty_cache()

Overwriting /kaggle/working/camo_m3fd_eval.py


In [19]:
import subprocess, os, time

def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4", "/kaggle/working/log_camo_0.txt")
p1 = launch(1, "python -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4", "/kaggle/working/log_camo_1.txt")
print("Camo-M3FD launched:", p0.pid, p1.pid)
time.sleep(12)
!ps aux | grep "[c]amo_m3fd_eval.py"
!cat /kaggle/working/log_camo_0.txt
!cat /kaggle/working/log_camo_1.txt

Camo-M3FD launched: 373 375
root         373  0.0  0.0   2816  1832 ?        Ss   13:41   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4 > /kaggle/working/log_camo_0.txt 2>&1
root         374 47.0  2.1 8742396 699804 ?      Dl   13:41   0:05 python3 -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4
root         375  0.0  0.0   2816  1960 ?        Ss   13:41   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4 > /kaggle/working/log_camo_1.txt 2>&1
root         376 46.3  2.1 8742264 699452 ?      Sl   13:41   0:05 python3 -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4


In [23]:
!ps aux | grep "[c]amo_m3fd_eval.py"
!tail -n 10 /kaggle/working/log_camo_0.txt /kaggle/working/log_camo_1.txt
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

root         373  0.0  0.0   2816  1832 ?        Ss   13:41   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4 > /kaggle/working/log_camo_0.txt 2>&1
root         374 81.5  7.2 13616252 2375992 ?    Rl   13:41  11:28 python3 -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4
root         375  0.0  0.0   2816  1960 ?        Ss   13:41   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4 > /kaggle/working/log_camo_1.txt 2>&1
root         376 81.5  8.2 14883640 2696184 ?    Rl   13:41  11:27 python3 -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4
==> /kaggle/working/log_camo_0.txt <==
100%|██████████| 44.7M/44.7M [00:00<00:00, 143MB/s] 
[Camo-M3FD | rgb_only       s0] test=0.0 (low=0.0, high=0.0, xover=0.0)
[Camo-M3FD | concat         s0] test=0.0 (low=0.0,

In [24]:
!python -u /kaggle/working/camo_m3fd_eval.py --report


=== Camo-M3FD (test) | Contrast gray-level q33=43.9, q66=60.9 (min=10.8, max=125.7) ===
rgb_only        n=5  overall=  0.0+-0.0  low_c=  0.0+-0.0  mid_c=  0.0+-0.0  high_c=  0.0+-0.0  crossover_realbg=  0.0+-0.0
concat          n=5  overall=  3.6+-3.2  low_c=  0.0+-0.0  mid_c=  3.1+-2.3  high_c=  7.6+-7.5  crossover_realbg=  0.1+-0.1
concat_drop     n=5  overall=  2.0+-2.6  low_c=  0.0+-0.0  mid_c=  1.6+-1.7  high_c=  4.3+-6.0  crossover_realbg=  0.1+-0.1
concat_bg_p25   n=5  overall=  1.4+-3.1  low_c=  0.0+-0.0  mid_c=  1.0+-2.3  high_c=  3.1+-7.0  crossover_realbg=  0.0+-0.0
concat_bg_p50   n=5  overall=  0.0+-0.0  low_c=  0.0+-0.0  mid_c=  0.0+-0.0  high_c=  0.0+-0.0  crossover_realbg=  0.0+-0.0
gate_raw        n=5  overall=  6.0+-3.6  low_c=  0.1+-0.3  mid_c=  7.3+-4.7  high_c= 10.6+-6.5  crossover_realbg=  0.1+-0.0
Welch p vs concat (low_c | crossover_realbg):
  rgb_only       : p_low=0.8943, p_xover=0.03264
  concat_drop    : p_low=0.3638, p_xover=0.2052
  concat_bg_p25  : p_low

In [25]:
import glob, os, numpy as np
from PIL import Image

base = '/kaggle/input/datasets/hvelesaca/camo-m3fd/train'
gts = sorted(glob.glob(f'{base}/GT/*'))[:50]
fg_fracs, uniqs = [], set()
for p in gts:
    a = np.asarray(Image.open(p).convert('L'))
    uniqs.update(np.unique(a).tolist()[:5])
    m = (a > 127) if a.max() > 1 else (a > 0)
    fg_fracs.append(m.mean() * 100)

print(f"Mask unique sample values: {sorted(list(uniqs))[:10]}")
print(f"Foreground area %: min={min(fg_fracs):.2f}%, median={np.median(fg_fracs):.2f}%, max={max(fg_fracs):.2f}%")
print("Sample sizes (RGB, Thermal, GT):",
      Image.open(gts[0].replace('/GT/', '/Imgs/')).size,
      Image.open(gts[0].replace('/GT/', '/Thermal/')).size,
      Image.open(gts[0]).size)

Mask unique sample values: [0, 255]
Foreground area %: min=0.02%, median=0.08%, max=0.39%
Sample sizes (RGB, Thermal, GT): (1024, 768) (1024, 768) (1024, 768)


In [26]:
%%writefile /kaggle/working/camo_m3fd_eval.py
import os, sys, glob, json, time, random, argparse
sys.path.insert(0, '/kaggle/working')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from scipy import stats

from phase1_rev import Net, iou_per_image, norm_rgb, norm_lw, descriptors, seed_all, DEV, AMP

CAMO_ROOT = '/kaggle/input/datasets/hvelesaca/camo-m3fd'
OUT_DIR = '/kaggle/working/phase1_camo_m3fd'
H_IMG, W_IMG = 512, 640  # native 4:3 aspect ratio for 1024x768

CONFIGS = {
    'rgb_only':      dict(fusion='none'),
    'concat':        dict(fusion='concat'),
    'concat_drop':   dict(fusion='concat', drop=0.25, mode='zero'),
    'concat_bg_p25': dict(fusion='concat', drop=0.25, mode='shuffle'),
    'concat_bg_p50': dict(fusion='concat', drop=0.50, mode='shuffle'),
    'gate_raw':      dict(fusion='gate'),
}

def tiny_seg_loss(logit, m):
    pw = torch.tensor([25.0], device=logit.device)
    bce = F.binary_cross_entropy_with_logits(logit, m, pos_weight=pw)
    p = torch.sigmoid(logit)
    inter = (p * m).sum((1, 2, 3))
    dice = 1.0 - (2.0 * inter + 1.0) / (p.sum((1, 2, 3)) + m.sum((1, 2, 3)) + 1.0)
    return 0.5 * bce + dice.mean()

def load_split(split):
    base = os.path.join(CAMO_ROOT, split)
    im_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Imgs/*'))}
    th_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Thermal/*'))}
    gt_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/GT/*'))}
    keys = sorted(set(im_map) & set(th_map) & set(gt_map))
    N = len(keys)
    rgb = np.empty((N, H_IMG, W_IMG, 3), np.uint8)
    th  = np.empty((N, H_IMG, W_IMG), np.uint8)
    msk = np.empty((N, H_IMG, W_IMG), np.uint8)
    for i, k in enumerate(keys):
        rgb[i] = np.asarray(Image.open(im_map[k]).convert('RGB').resize((W_IMG, H_IMG), Image.BILINEAR))
        th[i]  = np.asarray(Image.open(th_map[k]).convert('L').resize((W_IMG, H_IMG), Image.BILINEAR))
        ma = np.asarray(Image.open(gt_map[k]).convert('L').resize((W_IMG, H_IMG), Image.NEAREST))
        msk[i] = (ma > 127) if ma.max() > 1 else (ma > 0)
    return (torch.from_numpy(rgb).permute(0, 3, 1, 2).contiguous(),
            torch.from_numpy(th).unsqueeze(1).contiguous(),
            torch.from_numpy(msk).unsqueeze(1).contiguous())

@torch.no_grad()
def eval_split(net, cfg, data_tuple, bs=16):
    net.eval()
    te_r, te_t, te_m = data_tuple
    ious, cons, ious_shuf, alphas = [], [], [], []
    N = len(te_r)
    perm = torch.roll(torch.arange(N), shifts=max(1, N // 3))
    shuf_t = te_t[perm]

    for i in range(0, N, bs):
        rgb = te_r[i:i+bs].to(DEV).float() / 255
        lw  = te_t[i:i+bs].to(DEV).float() / 255
        lws = shuf_t[i:i+bs].to(DEV).float() / 255
        msk = te_m[i:i+bs].to(DEV).float()
        with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
            logit, alpha = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            logit_s, _   = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lws), lws)
        ious.append(iou_per_image(logit, msk).cpu())
        ious_shuf.append(iou_per_image(logit_s, msk).cpu())
        if alpha is not None:
            alphas.append(alpha.float().cpu())
        dil = (F.max_pool2d(msk, 31, 1, 15) > 0.5) & (msk < 0.5)
        for b in range(len(rgb)):
            fg = msk[b] > 0.5; rg = dil[b]
            c = abs(lw[b][fg].mean().item() - lw[b][rg].mean().item()) * 255.0 if (fg.any() and rg.any()) else 0.0
            cons.append(c)

    ious = 100 * torch.cat(ious).numpy()
    ious_shuf = 100 * torch.cat(ious_shuf).numpy()
    cons = np.array(cons)
    q33, q66 = np.quantile(cons, [0.33, 0.66])
    out = dict(
        overall=float(ious.mean()),
        low_c=float(ious[cons <= q33].mean()),
        mid_c=float(ious[(cons > q33) & (cons <= q66)].mean()),
        high_c=float(ious[cons > q66].mean()),
        crossover_realbg=float(ious_shuf.mean()),
        q33_gl=float(q33), q66_gl=float(q66),
        c_min=float(cons.min()), c_med=float(np.median(cons)), c_max=float(cons.max()),
    )
    if alphas:
        al = torch.cat(alphas).numpy()
        out['alpha_low_p3'] = float(al[cons <= q33, 0].mean())
        out['alpha_high_p3'] = float(al[cons > q66, 0].mean())
    return out

def train_and_eval(name, seed, tr, te, vt, epochs=35):
    cfg = CONFIGS[name]; seed_all(seed)
    net = Net(cfg['fusion'], arch='resnet18').to(DEV)
    tr_r, tr_t, tr_m = [x.to(DEV) for x in tr]
    if cfg['fusion'] == 'gate':
        with torch.no_grad():
            z = descriptors(tr_t[:256].float() / 255)
            net.mu.copy_(z.mean(0)); net.sd.copy_(z.std(0) + 1e-3)
    B, N = 8, len(tr_r); ipe = N // B
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=epochs * ipe)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    t0 = time.time()
    for ep in range(epochs):
        net.train(); perm = torch.randperm(N, device=DEV)
        for it in range(ipe):
            idx = perm[it*B:(it+1)*B]
            rgb = tr_r[idx].float() / 255; lw = tr_t[idx].float() / 255; msk = tr_m[idx].float()
            flip = (torch.rand(B, device=DEV) < 0.5).view(-1, 1, 1, 1)
            rgb = torch.where(flip, rgb.flip(-1), rgb); lw = torch.where(flip, lw.flip(-1), lw); msk = torch.where(flip, msk.flip(-1), msk)
            if cfg.get('drop', 0.0) > 0:
                drop = (torch.rand(B, device=DEV) < cfg['drop']).view(-1, 1, 1, 1)
                if cfg.get('mode') == 'zero':
                    lw = torch.where(drop, torch.zeros_like(lw), lw)
                else:
                    lw_shuf = torch.roll(lw, shifts=1, dims=0)
                    lw = torch.where(drop, lw_shuf, lw)
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, _ = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            loss = tiny_seg_loss(logit.float(), msk)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 5.0); scaler.step(opt); scaler.update(); sched.step()

    res = dict(cfg=name, seed=seed, min=round((time.time()-t0)/60, 2),
               test=eval_split(net, cfg, te), valtest=eval_split(net, cfg, vt))
    print(f'[Camo-M3FD | {name:14s} s{seed}] test={res["test"]["overall"]:.1f} '
          f'(low={res["test"]["low_c"]:.1f}, high={res["test"]["high_c"]:.1f}, xover={res["test"]["crossover_realbg"]:.1f})', flush=True)
    return res

def report():
    rows = [json.loads(l) for l in open(f'{OUT_DIR}/results.jsonl')]
    for split_key in ['test', 'valtest']:
        print(f'\n=== Camo-M3FD ({split_key}) | Contrast gray-level q33={rows[0][split_key]["q33_gl"]:.1f}, q66={rows[0][split_key]["q66_gl"]:.1f} (min={rows[0][split_key]["c_min"]:.1f}, max={rows[0][split_key]["c_max"]:.1f}) ===')
        M = {}
        for c in [k for k in CONFIGS if any(r['cfg'] == k for r in rows)]:
            rs = [r[split_key] for r in rows if r['cfg'] == c]
            M[c] = rs
            parts = []
            for m in ['overall', 'low_c', 'mid_c', 'high_c', 'crossover_realbg']:
                v = np.array([x[m] for x in rs])
                sd = v.std(ddof=1) if len(v) > 1 else 0.0
                parts.append(f'{m}={v.mean():5.1f}+-{sd:3.1f}')
            print(f'{c:15s} n={len(rs)}  ' + '  '.join(parts))
        if 'concat' in M and len(M['concat']) > 1:
            c_x = [x['crossover_realbg'] for x in M['concat']]
            c_l = [x['low_c'] for x in M['concat']]
            print('Welch p vs concat (low_c | crossover_realbg):')
            for c, rs in M.items():
                if c != 'concat' and len(rs) > 1:
                    _, pl = stats.ttest_ind([x['low_c'] for x in rs], c_l, equal_var=False)
                    _, px = stats.ttest_ind([x['crossover_realbg'] for x in rs], c_x, equal_var=False)
                    print(f'  {c:15s}: p_low={pl:.4g}, p_xover={px:.4g}')

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--configs', default=','.join(CONFIGS))
    ap.add_argument('--seeds', default='0,1,2,3,4')
    ap.add_argument('--epochs', type=int, default=35)
    ap.add_argument('--report', action='store_true')
    args = ap.parse_args()
    os.makedirs(OUT_DIR, exist_ok=True)
    if args.report:
        report(); sys.exit()
    tr, va, te = load_split('train'), load_split('val'), load_split('test')
    vt = tuple(torch.cat([va[i], te[i]], dim=0) for i in range(3))
    print(f'Loaded Camo-M3FD: train={len(tr[0])}, val={len(va[0])}, test={len(te[0])}', flush=True)
    res_path = f'{OUT_DIR}/results.jsonl'
    done = {(r['cfg'], r['seed']) for r in map(json.loads, open(res_path))} if os.path.exists(res_path) else set()
    for s in [int(x) for x in args.seeds.split(',')]:
        for c in args.configs.split(','):
            if (c, s) in done: continue
            r = train_and_eval(c, s, tr, te, vt, epochs=args.epochs)
            with open(res_path, 'a+') as f:
                f.seek(0); txt = f.read()
                if txt and not txt.endswith('\n'): f.write('\n')
                f.write(json.dumps(r) + '\n')
            torch.cuda.empty_cache()

Overwriting /kaggle/working/camo_m3fd_eval.py


In [27]:
!rm -f /kaggle/working/phase1_camo_m3fd/results.jsonl
!CUDA_VISIBLE_DEVICES=0 python -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat --seeds 0

Loaded Camo-M3FD: train=491, val=61, test=62
[Camo-M3FD | rgb_only       s0] test=23.6 (low=23.1, high=24.3, xover=23.6)
[Camo-M3FD | concat         s0] test=35.2 (low=34.8, high=35.5, xover=0.1)


In [28]:
import subprocess, os, time

def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4", "/kaggle/working/log_camo_0.txt")
p1 = launch(1, "python -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4", "/kaggle/working/log_camo_1.txt")
print("Camo-M3FD 5-seed sweep launched:", p0.pid, p1.pid)
time.sleep(25)
!ps aux | grep "[c]amo_m3fd_eval.py"
!tail -n 5 /kaggle/working/log_camo_0.txt /kaggle/working/log_camo_1.txt

Camo-M3FD 5-seed sweep launched: 470 472
root         470  0.0  0.0   2816  1968 ?        Ss   14:51   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4 > /kaggle/working/log_camo_0.txt 2>&1
root         471 98.6  3.6 9233876 1212496 ?     Rl   14:51   0:24 python3 -u /kaggle/working/camo_m3fd_eval.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4
root         472  0.0  0.0   2816  1984 ?        Ss   14:51   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4 > /kaggle/working/log_camo_1.txt 2>&1
root         473 98.7  3.6 9230796 1215888 ?     Rl   14:51   0:24 python3 -u /kaggle/working/camo_m3fd_eval.py --configs concat_bg_p25,concat_bg_p50,gate_raw --seeds 0,1,2,3,4
==> /kaggle/working/log_camo_0.txt <==

==> /kaggle/working/log_camo_1.txt <==


In [4]:
!ps aux | grep "[c]amo_m3fd_eval.py"
!tail -n 5 /kaggle/working/log_camo_0.txt /kaggle/working/log_camo_1.txt

==> /kaggle/working/log_camo_0.txt <==
[Camo-M3FD | concat         s3] test=32.7 (low=31.2, high=33.8, xover=0.1)
[Camo-M3FD | concat_drop    s3] test=31.1 (low=30.3, high=30.0, xover=5.5)
[Camo-M3FD | rgb_only       s4] test=23.7 (low=24.4, high=21.8, xover=23.7)
[Camo-M3FD | concat         s4] test=34.9 (low=32.1, high=37.4, xover=0.4)
[Camo-M3FD | concat_drop    s4] test=32.8 (low=32.7, high=34.6, xover=6.5)

==> /kaggle/working/log_camo_1.txt <==
[Camo-M3FD | gate_raw       s1] test=35.1 (low=33.8, high=35.8, xover=0.4)
[Camo-M3FD | concat_bg_p25  s2] test=29.7 (low=34.5, high=25.0, xover=22.6)
[Camo-M3FD | concat_bg_p50  s2] test=30.0 (low=33.8, high=27.4, xover=25.1)
[Camo-M3FD | gate_raw       s2] test=35.4 (low=33.0, high=37.4, xover=0.2)
[Camo-M3FD | concat_bg_p25  s3] test=31.1 (low=33.5, high=32.5, xover=21.1)


In [3]:
import json, os, subprocess, zipfile, glob
from IPython.display import FileLink, display

res_path = '/kaggle/working/phase1_camo_m3fd/results.jsonl'
cfgs = ['rgb_only', 'concat', 'concat_drop', 'concat_bg_p25', 'concat_bg_p50', 'gate_raw']
done = {(r['cfg'], r['seed']) for r in map(json.loads, open(res_path))} if os.path.exists(res_path) else set()
print(f"Completed Camo-M3FD runs: {len(done)} / 30")

missing_cfgs = sorted({c for c in cfgs for s in range(5) if (c, s) not in done})
missing_seeds = sorted({s for c in cfgs for s in range(5) if (c, s) not in done})
if missing_cfgs:
    print("Finishing remaining:", missing_cfgs, "seeds:", missing_seeds)
    cmd = f"CUDA_VISIBLE_DEVICES=0 python -u /kaggle/working/camo_m3fd_eval.py --configs {','.join(missing_cfgs)} --seeds {','.join(map(str, missing_seeds))}"
    subprocess.run(cmd, shell=True, check=True)
else:
    print("All 30 runs already complete!")

!python -u /kaggle/working/camo_m3fd_eval.py --report

with zipfile.ZipFile('/kaggle/working/bioai_ALL_final_v2.zip', 'w') as z:
    for f in (glob.glob('/kaggle/working/phase1_*/results.jsonl')
              + glob.glob('/kaggle/working/log_*.txt')
              + ['/kaggle/working/phase1_rev.py', '/kaggle/working/camo_m3fd_eval.py']):
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, '/kaggle/working'))
display(FileLink('bioai_ALL_final_v2.zip'))

Completed Camo-M3FD runs: 25 / 30
Finishing remaining: ['concat_bg_p25', 'concat_bg_p50', 'gate_raw'] seeds: [3, 4]
Loaded Camo-M3FD: train=491, val=61, test=62
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]


[Camo-M3FD | concat_bg_p50  s3] test=28.7 (low=32.9, high=27.1, xover=24.6)
[Camo-M3FD | gate_raw       s3] test=35.4 (low=34.1, high=34.9, xover=0.3)
[Camo-M3FD | concat_bg_p25  s4] test=29.2 (low=33.1, high=29.0, xover=21.1)
[Camo-M3FD | concat_bg_p50  s4] test=30.4 (low=33.4, high=29.3, xover=25.9)
[Camo-M3FD | gate_raw       s4] test=35.2 (low=33.3, high=35.3, xover=0.5)

=== Camo-M3FD (test) | Contrast gray-level q33=44.3, q66=62.7 (min=11.5, max=130.1) ===
rgb_only        n=5  overall= 23.9+-0.9  low_c= 24.4+-1.3  mid_c= 24.2+-1.5  high_c= 23.1+-1.2  crossover_realbg= 23.9+-0.9
concat          n=5  overall= 35.0+-1.7  low_c= 33.6+-2.3  mid_c= 34.6+-1.1  high_c= 36.8+-2.3  crossover_realbg=  0.3+-0.2
concat_drop     n=5  overall= 32.4+-1.2  low_c= 33.1+-1.8  mid_c= 32.1+-2.7  high_c= 31.9+-2.0  crossover_realbg=  6.2+-1.3
concat_bg_p25   n=5  overall= 29.7+-1.0  low_c= 32.7+-1.4  mid_c= 27.6+-1.7  high_c= 28.6+-2.8  crossover_realbg= 21.6+-0.6
concat_bg_p50   n=5  overall= 28.6+-1

/kaggle/working/bioai_ALL_final_v2.zip

In [5]:
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DELTAS = [8, 4, 2, 1, 0]

def load_jsonl(path):
    return [json.loads(l) for l in open(path)]

orig = load_jsonl('phase1_resnet18_orig/results.jsonl')
camo = load_jsonl('phase1_camo_m3fd/results.jsonl')

def stats_for(rows, cfg):
    A = np.array([[r['miou'][str(d)] for d in DELTAS] for r in rows if r['cfg'] == cfg])
    return A.mean(0), A.std(0, ddof=1) if len(A) > 1 else np.zeros(len(DELTAS))

# --- 1. fig_miou.pdf ---
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.4))
sel = [
    ('rgb_only', 'RGB only', 'k', '--'),
    ('concat', 'Concat', '#1f77b4', '-'),
    ('concat_d0_p10', 'dropout p=0.10', '#ff7f0e', '-'),
    ('concat_drop_d0', 'dropout p=0.25', '#2ca02c', '-'),
    ('concat_d0_p50', 'dropout p=0.50', '#d62728', '-'),
    ('gate_oracle', 'Oracle gate', '#9467bd', '-.'),
    ('oracle', 'Oracle switch', '#8c564b', ':'),
]
for cfg, lbl, c, ls in sel:
    m, s = stats_for(orig, cfg)
    for j in (0, 1):
        ax[j].errorbar(DELTAS, m, yerr=s, marker='o', markersize=3.5, capsize=2, lw=1.4, color=c, ls=ls, label=lbl)
ax[0].set_title('(a) All contrasts'); ax[0].set_ylabel('mIoU (%)')
ax[1].set_title('(b) Low contrast ($\\delta \\leq 1$ K)'); ax[1].set_xlim(1.15, -0.15); ax[1].set_ylim(40, 84)
for a in ax:
    a.invert_xaxis() if a == ax[0] else None
    a.set_xlabel('thermal contrast $\\delta$ (K)')
    a.grid(alpha=0.25)
ax[0].legend(fontsize=6.5, loc='lower left', frameon=True)
plt.tight_layout()
plt.savefig('fig_miou.pdf', dpi=300)
plt.close()

# --- 2. fig_tradeoff.pdf (cleanly spaced annotations, zero overlap!) ---
fig, ax = plt.subplots(figsize=(5.4, 3.8))
rgb_m, _ = stats_for(orig, 'rgb_only')
ax.axhline(rgb_m[-1], color='gray', ls='--', lw=1.2, label='RGB only')

pts = [
    ('concat', 'Concat', (6, -10)),
    ('concat_d0_p10', 'drop 0.10', (-42, 6)),
    ('concat_drop_d0', 'drop 0.25', (-44, 6)),
    ('concat_d0_p50', 'drop 0.50', (6, -4)),
    ('concat_drop', 'blank 0.25', (6, -8)),
    ('gate_raw', 'gate (unsup.)', (6, 6)),
    ('gate_sup', 'gate (sup.)', (-52, -10)),
    ('gate_oracle', 'oracle gate', (6, -4)),
]
for cfg, lbl, (dx, dy) in pts:
    m, s = stats_for(orig, cfg)
    x, y = m[3], m[4]      # delta=1, delta=0
    sx, sy = s[3], s[4]
    ax.errorbar(x, y, xerr=sx, yerr=sy, fmt='o', capsize=3, ms=5, lw=1.2)
    ax.annotate(lbl, (x, y), xytext=(dx, dy), textcoords='offset points', fontsize=8,
                arrowprops=dict(arrowstyle='-', lw=0.6, color='gray') if abs(dx) > 15 else None)
ax.set_xlabel('mIoU at $\\delta=1$ K (%)')
ax.set_ylabel('mIoU at $\\delta=0$ K (%)')
ax.set_xlim(73.5, 83.2); ax.set_ylim(41.2, 47.6)
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc='lower left')
plt.tight_layout()
plt.savefig('fig_tradeoff.pdf', dpi=300)
plt.close()

# --- 3. fig_alpha.pdf ---
fig, axs = plt.subplots(1, 3, figsize=(9.2, 2.8), sharey=True)
g_cfgs = [('gate_raw', 'Unsupervised gate'), ('gate_sup', 'Supervised gate'),
          ('gate_sup_drop', 'Sup. gate + blank drop'), ('gate_oracle', 'Oracle gate (true $\\delta$)')]
for st, ax in enumerate(axs):
    for cfg, lbl in g_cfgs:
        A = np.array([[r['alpha'][str(d)][st] for d in DELTAS] for r in orig if r['cfg'] == cfg and r.get('alpha')])
        ax.errorbar(DELTAS, A.mean(0), yerr=A.std(0, ddof=1), marker='o', ms=3.5, capsize=2, label=lbl)
    ax.set_title(f'P{st+3}'); ax.invert_xaxis(); ax.set_xlabel('$\\delta$ (K)'); ax.grid(alpha=0.25)
axs[0].set_ylabel(r'mean thermal gate $\bar{\alpha}_{ir}$')
axs[0].legend(fontsize=6.5, loc='lower left')
plt.tight_layout()
plt.savefig('fig_alpha.pdf', dpi=300)
plt.close()

# --- 4. fig_camo_m3fd.pdf ---
fig, axs = plt.subplots(1, 2, figsize=(9.0, 3.2), sharey=True)
c_order = [('rgb_only', 'RGB\nonly'), ('concat', 'Concat'), ('concat_drop', 'Blank\ndrop'),
           ('concat_bg_p25', 'Clutter\np=0.25'), ('concat_bg_p50', 'Clutter\np=0.50'), ('gate_raw', 'Gate\nunsup.')]
x = np.arange(len(c_order)); w = 0.36
for j, (sk, title) in enumerate([('test', 'Official Test (N=62)'), ('valtest', 'Val+Test (N=123)')]):
    all_m, all_s, xo_m, xo_s = [], [], [], []
    for cfg, _ in c_order:
        va = [r[sk]['overall'] for r in camo if r['cfg'] == cfg]
        vx = [r[sk]['crossover_realbg'] for r in camo if r['cfg'] == cfg]
        all_m.append(np.mean(va)); all_s.append(np.std(va, ddof=1))
        xo_m.append(np.mean(vx)); xo_s.append(np.std(vx, ddof=1))
    axs[j].bar(x - w/2, all_m, w, yerr=all_s, capsize=2.5, label='Normal Thermal', color='#4c72b0')
    axs[j].bar(x + w/2, xo_m,  w, yerr=xo_s,  capsize=2.5, label='Real-BG Crossover', color='#dd8452')
    axs[j].set_xticks(x); axs[j].set_xticklabels([l for _, l in c_order], fontsize=7.5)
    axs[j].set_title(title); axs[j].grid(axis='y', alpha=0.25)
axs[0].set_ylabel('mIoU (%)'); axs[0].legend(fontsize=7.5, loc='upper right')
plt.tight_layout()
plt.savefig('fig_camo_m3fd.pdf', dpi=300)
plt.close()
print("All 4 figures saved cleanly!")

All 4 figures saved cleanly!


## Now, Re-Checking the revised paper

In [6]:
%%writefile /kaggle/working/camo_m3fd_erased.py
import os, sys, glob, json, time, random, argparse
import cv2
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvm
from PIL import Image
from scipy import stats

CAMO_ROOT = '/kaggle/input/datasets/hvelesaca/camo-m3fd'
OUT_DIR = '/kaggle/working/phase1_camo_erased'
H_IMG, W_IMG = 512, 640
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEV == 'cuda'
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

CONFIGS = {
    'rgb_only':      dict(fusion='none'),
    'concat':        dict(fusion='concat'),
    'concat_drop':   dict(fusion='concat', drop=0.25, mode='zero'),
    'concat_er_p25': dict(fusion='concat', drop=0.25, mode='erased'),
    'concat_er_p50': dict(fusion='concat', drop=0.50, mode='erased'),
    'gate_raw':      dict(fusion='gate'),
}

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def norm_rgb(x): return (x - MEAN.to(x.device)) / STD.to(x.device)
def norm_lw(lw): return norm_rgb(lw.repeat(1, 3, 1, 1))
def _rp(x, p): return F.pad(x, (p, p, p, p), mode='replicate')

def descriptors(lw):
    x = lw.float()
    k = torch.ones(1, 1, 7, 7, device=x.device) / 49.
    m = F.conv2d(_rp(x, 3), k); v = (F.conv2d(_rp(x * x, 3), k) - m * m).clamp_min(1e-10)
    lstd = v.sqrt().mean((1, 2, 3))
    sx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]], device=x.device).view(1, 1, 3, 3)
    sy = sx.transpose(2, 3); xp = _rp(x, 1)
    g = torch.sqrt(F.conv2d(xp, sx) ** 2 + F.conv2d(xp, sy) ** 2 + 1e-10)
    hp = (x - F.avg_pool2d(_rp(x, 2), 5, 1)).abs().mean((1, 2, 3))
    return torch.log(torch.stack([lstd, g.mean((1, 2, 3)), hp, torch.quantile(g.flatten(1), 0.99, dim=1)], 1) + 1e-5)

def feats(bb, x):
    x = bb.maxpool(bb.relu(bb.bn1(bb.conv1(x))))
    x = bb.layer1(x); f3 = bb.layer2(x); f4 = bb.layer3(f3); f5 = bb.layer4(f4)
    return [f3, f4, f5]

class Net(nn.Module):
    def __init__(self, fusion):
        super().__init__()
        self.fusion = fusion
        chans = [128, 256, 512]
        self.rgb = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT); self.rgb.fc = nn.Identity()
        if fusion != 'none':
            self.ir = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT); self.ir.fc = nn.Identity()
            self.mix = nn.ModuleList([nn.Sequential(nn.Conv2d(2 * c, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(True)) for c in chans])
        if fusion == 'gate':
            self.register_buffer('mu', torch.zeros(4)); self.register_buffer('sd', torch.ones(4))
            self.gate = nn.Sequential(nn.Linear(4, 32), nn.GELU(), nn.Linear(32, 3))
            nn.init.zeros_(self.gate[2].weight); nn.init.zeros_(self.gate[2].bias)
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in chans])
        self.head = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True))
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, rgb_n, lw_n=None, lw_raw=None):
        fr = feats(self.rgb, rgb_n)
        alpha = None
        if self.fusion != 'none':
            fi = feats(self.ir, lw_n)
            if self.fusion == 'gate':
                with torch.autocast(DEV, enabled=False):
                    alpha = torch.sigmoid(self.gate((descriptors(lw_raw) - self.mu) / self.sd))
            fr = [self.mix[i](torch.cat([fr[i], fi[i] * (1. if alpha is None else alpha[:, i].view(-1, 1, 1, 1).to(fi[i].dtype))], 1))
                  for i in range(3)]
        s = fr[0].shape[-2:]
        y = sum(F.interpolate(l(f), size=s, mode='bilinear', align_corners=False) for l, f in zip(self.lat, fr))
        return F.interpolate(self.out(self.head(y)), size=rgb_n.shape[-2:], mode='bilinear', align_corners=False), alpha

def tiny_seg_loss(logit, m):
    pw = torch.tensor([25.0], device=logit.device)
    bce = F.binary_cross_entropy_with_logits(logit, m, pos_weight=pw)
    p = torch.sigmoid(logit)
    inter = (p * m).sum((1, 2, 3))
    dice = 1.0 - (2.0 * inter + 1.0) / (p.sum((1, 2, 3)) + m.sum((1, 2, 3)) + 1.0)
    return 0.5 * bce + dice.mean()

def iou_per_image(logit, m):
    pred = (torch.sigmoid(logit.float()) > 0.5).float()
    inter = (pred * m).sum((1, 2, 3)); union = (pred + m - pred * m).sum((1, 2, 3))
    return (inter + 1e-6) / (union + 1e-6)

def erase_target_thermal(th_u8, msk_u8, rng):
    """Inpaint 3px-dilated target mask using surrounding background + matched sensor grain."""
    dil = cv2.dilate(msk_u8, np.ones((7, 7), np.uint8), iterations=1)
    inp = cv2.inpaint(th_u8, dil, inpaintRadius=5, flags=cv2.INPAINT_TELEA).astype(np.float32)
    ring = (cv2.dilate(msk_u8, np.ones((31, 31), np.uint8), iterations=1) > 0) & (dil == 0)
    hp = th_u8.astype(np.float32) - cv2.GaussianBlur(th_u8.astype(np.float32), (5, 5), 1.0)
    sig = float(np.clip(np.std(hp[ring]) if ring.sum() > 20 else 1.5, 0.5, 4.0))
    noise = rng.normal(0.0, sig, size=inp.shape).astype(np.float32)
    out = np.where(dil > 0, np.clip(inp + noise, 0, 255), th_u8.astype(np.float32))
    return np.round(out).astype(np.uint8)

def load_split(split):
    base = os.path.join(CAMO_ROOT, split)
    im_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Imgs/*'))}
    th_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/Thermal/*'))}
    gt_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sorted(glob.glob(f'{base}/GT/*'))}
    keys = sorted(set(im_map) & set(th_map) & set(gt_map))
    N = len(keys)
    rgb = np.empty((N, H_IMG, W_IMG, 3), np.uint8)
    th  = np.empty((N, H_IMG, W_IMG), np.uint8)
    the = np.empty((N, H_IMG, W_IMG), np.uint8)
    msk = np.empty((N, H_IMG, W_IMG), np.uint8)
    rng = np.random.default_rng(12345)
    for i, k in enumerate(keys):
        rgb[i] = np.asarray(Image.open(im_map[k]).convert('RGB').resize((W_IMG, H_IMG), Image.BILINEAR))
        t_u8   = np.asarray(Image.open(th_map[k]).convert('L').resize((W_IMG, H_IMG), Image.BILINEAR))
        ma     = np.asarray(Image.open(gt_map[k]).convert('L').resize((W_IMG, H_IMG), Image.NEAREST))
        m_u8   = ((ma > 127) if ma.max() > 1 else (ma > 0)).astype(np.uint8)
        th[i]  = t_u8
        msk[i] = m_u8
        the[i] = erase_target_thermal(t_u8, m_u8, rng)
    return (torch.from_numpy(rgb).permute(0, 3, 1, 2).contiguous(),
            torch.from_numpy(th).unsqueeze(1).contiguous(),
            torch.from_numpy(the).unsqueeze(1).contiguous(),
            torch.from_numpy(msk).unsqueeze(1).contiguous())

@torch.no_grad()
def eval_split(net, cfg, data_tuple, bs=16):
    net.eval()
    te_r, te_t, te_e, te_m = data_tuple
    ious, ious_er, cons, alphas = [], [], [], []
    for i in range(0, len(te_r), bs):
        rgb = te_r[i:i+bs].to(DEV).float() / 255
        lw  = te_t[i:i+bs].to(DEV).float() / 255
        lwe = te_e[i:i+bs].to(DEV).float() / 255
        msk = te_m[i:i+bs].to(DEV).float()
        with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
            logit, alpha = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            logit_e, _   = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lwe), lwe)
        ious.append(iou_per_image(logit, msk).cpu())
        ious_er.append(iou_per_image(logit_e, msk).cpu())
        if alpha is not None: alphas.append(alpha.float().cpu())
        dil = (F.max_pool2d(msk, 31, 1, 15) > 0.5) & (msk < 0.5)
        for b in range(len(rgb)):
            fg = msk[b] > 0.5; rg = dil[b]
            cons.append(abs(lw[b][fg].mean().item() - lw[b][rg].mean().item()) * 255.0 if (fg.any() and rg.any()) else 0.0)

    ious = 100 * torch.cat(ious).numpy()
    ious_er = 100 * torch.cat(ious_er).numpy()
    cons = np.array(cons)
    q33, q66 = np.quantile(cons, [0.33, 0.66])
    out = dict(
        overall=float(ious.mean()),
        low_c=float(ious[cons <= q33].mean()),
        mid_c=float(ious[(cons > q33) & (cons <= q66)].mean()),
        high_c=float(ious[cons > q66].mean()),
        erased_xover=float(ious_er.mean()),
        q33_gl=float(q33), q66_gl=float(q66),
    )
    if alphas:
        al = torch.cat(alphas).numpy()
        out['alpha_low_p3'] = float(al[cons <= q33, 0].mean())
        out['alpha_high_p3'] = float(al[cons > q66, 0].mean())
    return out

def train_and_eval(name, seed, tr, te, vt, epochs=35):
    cfg = CONFIGS[name]; seed_all(seed)
    net = Net(cfg['fusion']).to(DEV)
    tr_r, tr_t, tr_e, tr_m = [x.to(DEV) for x in tr]
    if cfg['fusion'] == 'gate':
        with torch.no_grad():
            z = descriptors(tr_t[:256].float() / 255)
            net.mu.copy_(z.mean(0)); net.sd.copy_(z.std(0) + 1e-3)
    B, N = 8, len(tr_r); ipe = N // B
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=epochs * ipe)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    t0 = time.time()
    for ep in range(epochs):
        net.train(); perm = torch.randperm(N, device=DEV)
        for it in range(ipe):
            idx = perm[it*B:(it+1)*B]
            rgb = tr_r[idx].float() / 255; lw = tr_t[idx].float() / 255; lwe = tr_e[idx].float() / 255; msk = tr_m[idx].float()
            flip = (torch.rand(B, device=DEV) < 0.5).view(-1, 1, 1, 1)
            rgb = torch.where(flip, rgb.flip(-1), rgb); lw = torch.where(flip, lw.flip(-1), lw)
            lwe = torch.where(flip, lwe.flip(-1), lwe); msk = torch.where(flip, msk.flip(-1), msk)
            if cfg.get('drop', 0.0) > 0:
                drop = (torch.rand(B, device=DEV) < cfg['drop']).view(-1, 1, 1, 1)
                lw = torch.where(drop, torch.zeros_like(lw) if cfg.get('mode') == 'zero' else lwe, lw)
            with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
                logit, _ = net(norm_rgb(rgb), None if cfg['fusion'] == 'none' else norm_lw(lw), lw)
            loss = tiny_seg_loss(logit.float(), msk)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 5.0); scaler.step(opt); scaler.update(); sched.step()

    res = dict(cfg=name, seed=seed, min=round((time.time()-t0)/60, 2),
               test=eval_split(net, cfg, te), valtest=eval_split(net, cfg, vt))
    print(f'[Camo-Erased | {name:14s} s{seed}] test={res["test"]["overall"]:.1f} '
          f'(low={res["test"]["low_c"]:.1f}, high={res["test"]["high_c"]:.1f}, erased_xover={res["test"]["erased_xover"]:.1f})', flush=True)
    return res

def report():
    rows = [json.loads(l) for l in open(f'{OUT_DIR}/results.jsonl')]
    for sk in ['test', 'valtest']:
        print(f'\n=== Camo-M3FD Target-Erased ({sk}) ===')
        M = {}
        for c in [k for k in CONFIGS if any(r['cfg'] == k for r in rows)]:
            rs = [r[sk] for r in rows if r['cfg'] == c]; M[c] = rs
            parts = [f'{m}={np.mean([x[m] for x in rs]):5.1f}+-{np.std([x[m] for x in rs], ddof=1):3.1f}'
                     for m in ['overall', 'low_c', 'mid_c', 'high_c', 'erased_xover']]
            print(f'{c:15s} n={len(rs)}  ' + '  '.join(parts))
        c_e = [x['erased_xover'] for x in M['concat']]
        print('Welch p vs concat (erased_xover):')
        for c, rs in M.items():
            if c != 'concat':
                _, px = stats.ttest_ind([x['erased_xover'] for x in rs], c_e, equal_var=False)
                print(f'  {c:15s}: p_erased={px:.4g}')

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--configs', default=','.join(CONFIGS))
    ap.add_argument('--seeds', default='0,1,2,3,4')
    ap.add_argument('--epochs', type=int, default=35)
    ap.add_argument('--report', action='store_true')
    args = ap.parse_args()
    os.makedirs(OUT_DIR, exist_ok=True)
    if args.report: report(); sys.exit()
    tr, va, te = load_split('train'), load_split('val'), load_split('test')
    vt = tuple(torch.cat([va[i], te[i]], dim=0) for i in range(4))
    print(f'Loaded Camo-M3FD (with target-erased): train={len(tr[0])}, val={len(va[0])}, test={len(te[0])}', flush=True)
    res_path = f'{OUT_DIR}/results.jsonl'
    done = {(r['cfg'], r['seed']) for r in map(json.loads, open(res_path))} if os.path.exists(res_path) else set()
    for s in [int(x) for x in args.seeds.split(',')]:
        for c in args.configs.split(','):
            if (c, s) in done: continue
            r = train_and_eval(c, s, tr, te, vt, epochs=args.epochs)
            with open(res_path, 'a+') as f:
                f.seek(0); txt = f.read()
                if txt and not txt.endswith('\n'): f.write('\n')
                f.write(json.dumps(r) + '\n')
            torch.cuda.empty_cache()

Writing /kaggle/working/camo_m3fd_erased.py


In [7]:
import subprocess, os, time

def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/camo_m3fd_erased.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4", "/kaggle/working/log_er_0.txt")
p1 = launch(1, "python -u /kaggle/working/camo_m3fd_erased.py --configs concat_er_p25,concat_er_p50,gate_raw --seeds 0,1,2,3,4", "/kaggle/working/log_er_1.txt")
print("Launched:", p0.pid, p1.pid)
time.sleep(35)
!ps aux | grep "[c]amo_m3fd_erased.py"
!tail -n 5 /kaggle/working/log_er_0.txt /kaggle/working/log_er_1.txt

Launched: 155 157
root         155  0.0  0.0   2816  1892 ?        Ss   18:31   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_erased.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4 > /kaggle/working/log_er_0.txt 2>&1
root         156 98.5  4.4 10027284 1455484 ?    Rl   18:31   0:34 python3 -u /kaggle/working/camo_m3fd_erased.py --configs rgb_only,concat,concat_drop --seeds 0,1,2,3,4
root         157  0.0  0.0   2816  1960 ?        Ss   18:31   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_erased.py --configs concat_er_p25,concat_er_p50,gate_raw --seeds 0,1,2,3,4 > /kaggle/working/log_er_1.txt 2>&1
root         158 99.2  4.3 10024600 1442244 ?    Rl   18:31   0:34 python3 -u /kaggle/working/camo_m3fd_erased.py --configs concat_er_p25,concat_er_p50,gate_raw --seeds 0,1,2,3,4
==> /kaggle/working/log_er_0.txt <==

==> /kaggle/working/log_er_1.txt <==


In [16]:
!ps aux | grep "[c]amo_m3fd_erased.py"
!wc -l /kaggle/working/phase1_camo_erased/results.jsonl 2>/dev/null || true
!tail -n 5 /kaggle/working/log_er_0.txt /kaggle/working/log_er_1.txt

root         228  0.0  0.0   2816  1984 ?        Ss   20:15   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_erased.py --configs concat_sm_p25 --seeds 0,1,2,3,4 > /kaggle/working/log_sm_0.txt 2>&1
root         229  0.0  0.0   2816  1840 ?        Ss   20:15   0:00 /bin/sh -c python -u /kaggle/working/camo_m3fd_erased.py --configs concat_sm_p50 --seeds 0,1,2,3,4 > /kaggle/working/log_sm_1.txt 2>&1
root         230 99.9 10.0 16844712 3305720 ?    Rl   20:15  10:18 python3 -u /kaggle/working/camo_m3fd_erased.py --configs concat_sm_p25 --seeds 0,1,2,3,4
root         231 99.9 10.0 16844820 3305704 ?    Rl   20:15  10:18 python3 -u /kaggle/working/camo_m3fd_erased.py --configs concat_sm_p50 --seeds 0,1,2,3,4
22 /kaggle/working/phase1_camo_erased/results.jsonl
==> /kaggle/working/log_er_0.txt <==
[Camo-Erased | concat         s3] test=34.7 (low=32.8, high=35.6, erased_xover=0.7)
[Camo-Erased | concat_drop    s3] test=29.3 (low=30.0, high=27.9, erased_xover=8.1)
[Camo-Erased | rgb_only    

In [13]:
import json, os, subprocess, time

# 1. Keep clean configs (rgb_only, concat, concat_drop, gate_raw); remove leaky concat_er_*
res_path = '/kaggle/working/phase1_camo_erased/results.jsonl'
clean = [r for r in map(json.loads, open(res_path)) if not r['cfg'].startswith('concat_er_')]
with open(res_path, 'w') as f:
    for r in clean:
        f.write(json.dumps(r) + '\n')
print(f"Kept {len(clean)} clean runs (rgb_only, concat, concat_drop, gate_raw).")

# 2. Patch camo_m3fd_erased.py to add leak-free smooth-background dropout (zero GT access!)
code = open('/kaggle/working/camo_m3fd_erased.py').read()
old_cfg = "    'concat_er_p50': dict(fusion='concat', drop=0.50, mode='erased'),"
new_cfg = (old_cfg + "\n"
           "    'concat_sm_p25': dict(fusion='concat', drop=0.25, mode='smooth'),\n"
           "    'concat_sm_p50': dict(fusion='concat', drop=0.50, mode='smooth'),")
assert old_cfg in code
code = code.replace(old_cfg, new_cfg, 1)

old_drop = "lw = torch.where(drop, torch.zeros_like(lw) if cfg.get('mode') == 'zero' else lwe, lw)"
new_drop = (
    "if cfg.get('mode') == 'zero':\n"
    "                    lw = torch.where(drop, torch.zeros_like(lw), lw)\n"
    "                elif cfg.get('mode') == 'smooth':\n"
    "                    sm = F.interpolate(F.adaptive_avg_pool2d(lw, (12, 16)), size=lw.shape[-2:], mode='bicubic', align_corners=False)\n"
    "                    sm = (sm + 0.01 * torch.randn_like(sm)).clamp(0, 1)\n"
    "                    lw = torch.where(drop, sm, lw)\n"
    "                else:\n"
    "                    lw = torch.where(drop, lwe, lw)"
)
assert old_drop in code
code = code.replace(old_drop, new_drop, 1)
open('/kaggle/working/camo_m3fd_erased.py', 'w').write(code)
print("Patched camo_m3fd_erased.py with leak-free smooth-background dropout.")

# 3. Launch only the 2 clean dropout configs in parallel (~15 min on T4x2)
def launch(gpu, cmd, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, env=env, start_new_session=True)

p0 = launch(0, "python -u /kaggle/working/camo_m3fd_erased.py --configs concat_sm_p25 --seeds 0,1,2,3,4", "/kaggle/working/log_sm_0.txt")
p1 = launch(1, "python -u /kaggle/working/camo_m3fd_erased.py --configs concat_sm_p50 --seeds 0,1,2,3,4", "/kaggle/working/log_sm_1.txt")
print("Launched clean dropout sweep:", p0.pid, p1.pid)

Kept 20 clean runs (rgb_only, concat, concat_drop, gate_raw).
Patched camo_m3fd_erased.py with leak-free smooth-background dropout.
Launched clean dropout sweep: 228 229


In [15]:
!python -u /kaggle/working/camo_m3fd_erased.py --report


=== Camo-M3FD Target-Erased (test) ===
rgb_only        n=5  overall= 23.4+-1.0  low_c= 24.2+-1.8  mid_c= 23.4+-1.3  high_c= 22.7+-2.6  erased_xover= 23.4+-1.0
concat          n=5  overall= 34.1+-0.5  low_c= 31.4+-1.4  mid_c= 35.7+-1.1  high_c= 35.2+-1.2  erased_xover=  1.2+-0.4
concat_drop     n=5  overall= 31.9+-2.0  low_c= 32.0+-2.6  mid_c= 32.1+-1.9  high_c= 31.4+-2.5  erased_xover=  7.6+-1.4
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
concat_sm_p25   n=1  overall= 32.6+-nan  low_c= 32.5+-nan  mid_c= 32.7+-nan  high_c= 32.5+-nan  erased_xover=  9.7+-nan
concat_sm_p50   n=1  overall= 29.7+-nan  low_c= 34.3+-nan  mid_c= 24.5+-nan  high_c= 30.1+-nan  erased_xover= 19.1+-nan
gate_raw        